In [ ]:
from __future__ import annotations

import json
import os
import re
import sys
from datetime import datetime
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

DEFAULT_PROJECT_ROOT = Path(r"C:/Users/Haady/Documents/Antigravity/Dataverwerking_Antigravity")
PROJECT_ROOT = Path(os.environ.get("DATAVERWERKING_PROJECT_ROOT", DEFAULT_PROJECT_ROOT)).resolve()
if not (PROJECT_ROOT / "thesis").exists() and (Path.cwd() / "thesis").exists():
    PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

NOTEBOOK_CREATED_AT = "2026-06-02"
NOTEBOOK_UPDATED_AT = "2026-06-05"
DATA_PEILDATUM = "2026-06-05"
INCLUDE_ENGLISH = False
RUN_ADVIES_LIVE = True
USE_FROZEN_ADVIES = True

DV2_BRONNEN_DIR = PROJECT_ROOT / "thesis" / "Analyse" / "DV2_validatie_bronnen"
DV1_BUNDLE_DIR = DV2_BRONNEN_DIR / "01_DV1_corpuscontext_20260603"
KABINET_BATCH_BUNDLE_DIR = DV2_BRONNEN_DIR / "02_kabinetsreactie_batch_capfix_full_20260603"
KABINET_HISTORISCHE_BUNDLE_DIR = DV2_BRONNEN_DIR / "03_kabinetsreactie_historische_39_batch_20260601"
ADVIES_GOLDEN_DIR = DV2_BRONNEN_DIR / "04_adviesextractie_golden_set_20260602"
RELIABILITY_BUNDLE_DIR = DV2_BRONNEN_DIR / "05_reliability_test_hertest"
KABINET_GOLDEN_DIR = DV2_BRONNEN_DIR / "06_kabinetsreactie_golden_set_20260604"
OLD_REFERENCES_BUNDLE_DIR = DV2_BRONNEN_DIR / "07_oude_claim_referenties"
CLASSMETA_CONSISTENCY_BUNDLE_DIR = DV2_BRONNEN_DIR / "08_classificatie_metadata_reproduceerbaarheid_20260507"
ADVIES_RELIABILITY_DIR = DV2_BRONNEN_DIR / "09_adviesextractie_reliability_test_hertest"

ADVIESVALIDATIE_FROZEN_DIR = DV2_BRONNEN_DIR / "01b_adviesvalidatie_171_frozen_20260605"
KABINET_171_FROZEN_DIR = DV2_BRONNEN_DIR / "02b_kabinetsreactie_171_frozen_20260605"
USE_171_SCOPE_KABINET = True

DV1_OUTPUT_DIR = DV1_BUNDLE_DIR
DV1_COLLEGE_DEFINITIES_CSV = DV1_OUTPUT_DIR / "05b_college_telling_definities.csv"
DV1_ANALYSEPOPULATIES_CSV = DV1_OUTPUT_DIR / "12c_corpus_vs_analysepopulaties.csv"
DV1_CORPUS_COLLEGES_EXPECTED = 171

KABINET_ANALYSE_BATCH_DIR = KABINET_BATCH_BUNDLE_DIR
EXPECTED_KABINET_RESULT_JSONS = 421
ALLOW_SMALL_KABINET_BATCH = False
KABINET_BATCH_DIR = KABINET_ANALYSE_BATCH_DIR

KABINET_HISTORISCHE_VALIDATIE_BATCH_DIR = KABINET_HISTORISCHE_BUNDLE_DIR
EXPECTED_HISTORISCHE_KABINET_RESULT_JSONS = 39

THESIS_TEXT_PATH = PROJECT_ROOT / "master_thesis_conceptversie.tex"
THESIS_INDEX_PATH = PROJECT_ROOT / "thesis_validatie_materiaal" / "00_INDEX.md"
OLD_ADVIES_SUMMARY_JSON = OLD_REFERENCES_BUNDLE_DIR / "oude_advies_summary_20260601_030050.json"
OLD_ADVIES_REPORT_MD = OLD_REFERENCES_BUNDLE_DIR / "oude_advies_report_20260601_030050.md"
OLD_KABINET_SUMMARY_JSON = OLD_REFERENCES_BUNDLE_DIR / "oude_kabinet_summary_validatie_run.json"
OLD_KABINET_STATS_JSON = OLD_REFERENCES_BUNDLE_DIR / "oude_kabinet_dataset_statistiek_validatie_run.json"
OLD_KABINET_REPORT_MD = OLD_REFERENCES_BUNDLE_DIR / "oude_kabinet_validatie_rapport.md"

RELIABILITY_JSONS = {
    "parallel_10x3": RELIABILITY_BUNDLE_DIR / "kabinetsreactie_parallel_10x3_reliability_summary.json",
    "sequentieel_4907": RELIABILITY_BUNDLE_DIR / "kabinetsreactie_sequentieel_4907_reliability_summary.json",
    "seed_4907": RELIABILITY_BUNDLE_DIR / "kabinetsreactie_seed_4907_reliability_summary.json",
}

CONSISTENCY_CLASSMETA_DIR = CLASSMETA_CONSISTENCY_BUNDLE_DIR
CONSISTENCY_CLASSMETA_EXPECTED_DOCS = 100
CONSISTENCY_CLASSMETA_EXPECTED_RUNS = 5

plt.rcParams["figure.figsize"] = (9, 4)
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 140)

In [ ]:
def read_json(path: Path) -> dict[str, Any]:
    if not path.exists():
        return {"_missing": str(path)}
    return json.loads(path.read_text(encoding="utf-8"))


def read_text(path: Path) -> str:
    if not path.exists():
        return ""
    return path.read_text(encoding="utf-8", errors="replace")


def count_result_jsons(batch_dir: Path) -> int:
    cases_dir = batch_dir / "cases"
    if cases_dir.is_dir():
        return len(list(cases_dir.glob("*/*_result.json")))
    return len(list(batch_dir.glob("**/*_result.json")))


def modified_at(path: Path) -> str | None:
    if not path.exists():
        return None
    return datetime.fromtimestamp(path.stat().st_mtime).strftime("%Y-%m-%d %H:%M:%S")


def count_csv_data_rows(path: Path) -> int | None:
    if not path.exists():
        return None
    with path.open("r", encoding="utf-8-sig", errors="replace") as handle:
        return max(sum(1 for _ in handle) - 1, 0)


def folder_file_count(path: Path) -> int | None:
    if not path.exists():
        return None
    if path.is_file():
        return 1
    return len([item for item in path.rglob("*") if item.is_file()])


def folder_total_mb(path: Path) -> float | None:
    if not path.exists():
        return None
    files = [path] if path.is_file() else [item for item in path.rglob("*") if item.is_file()]
    return round(sum(item.stat().st_size for item in files) / 1_000_000, 2)


def source_registry_row(naam: str, pad: Path, soort: str, dynamiek: str, rol: str) -> dict[str, Any]:
    return {
        "bron": naam,
        "soort": soort,
        "dynamiek": dynamiek,
        "rol": rol,
        "bestaat": pad.exists(),
        "bestanden": folder_file_count(pad),
        "mb": folder_total_mb(pad),
        "gewijzigd_op": modified_at(pad),
        "pad": str(pad),
    }


def dataset_input_row(naam: str, pad: Path, type_: str, rol: str) -> dict[str, Any]:
    row = {
        "naam": naam,
        "rol": rol,
        "type": type_,
        "bestaat": pad.exists(),
        "pad": str(pad),
        "gewijzigd_op": modified_at(pad),
        "result_jsons": None,
        "batch_summary": None,
    }
    if type_ == "batchmap":
        row["result_jsons"] = count_result_jsons(pad) if pad.exists() else 0
        row["batch_summary"] = (pad / "batch_summary.json").exists()
    return row


def label_counts(labels: pd.DataFrame, label_col: str = "label") -> pd.DataFrame:
    if labels is None or labels.empty or label_col not in labels.columns:
        return pd.DataFrame(columns=["label", "aantal", "percentage"])
    counts = labels[label_col].value_counts(dropna=False).rename_axis("label").reset_index(name="aantal")
    totaal = int(counts["aantal"].sum()) or 1
    counts["percentage"] = (counts["aantal"] / totaal * 100).round(1)
    order = {"groen": 0, "oranje": 1, "rood": 2}
    counts["_order"] = counts["label"].map(order).fillna(99)
    return counts.sort_values(["_order", "label"]).drop(columns="_order")


def value_counts_table(df: pd.DataFrame, column: str, name: str | None = None) -> pd.DataFrame:
    if df is None or df.empty or column not in df.columns:
        return pd.DataFrame(columns=[name or column, "aantal", "percentage"])
    table = df[column].fillna("<leeg>").value_counts().rename_axis(name or column).reset_index(name="aantal")
    totaal = int(table["aantal"].sum()) or 1
    table["percentage"] = (table["aantal"] / totaal * 100).round(1)
    return table


def _entropy_status(normalized_entropy: float | None, top_pct: float | None, n_labels: int) -> str:
    if normalized_entropy is None or top_pct is None or n_labels <= 1:
        return "NIET_BEREKEND"
    if top_pct >= 85 or normalized_entropy < 0.40:
        return "WAARSCHUWING"
    if top_pct >= 70 or normalized_entropy < 0.60:
        return "LET_OP"
    return "OK"


def label_entropy_record(
    naam: str,
    values: Any,
    expected_labels: list[str] | None = None,
) -> dict[str, Any]:
    series = pd.Series(values).dropna().astype(str)
    series = series[series.str.len() > 0]
    observed_labels = sorted(series.unique().tolist())
    labels = list(dict.fromkeys((expected_labels or []) + observed_labels))
    counts = series.value_counts().reindex(labels, fill_value=0) if labels else pd.Series(dtype=int)
    total = int(counts.sum())
    if total == 0:
        return {
            "test": naam,
            "status": "NIET_BEREKEND",
            "n": 0,
            "aantal_labels": 0,
            "top_label": None,
            "top_pct": None,
            "shannon_entropy": None,
            "genormaliseerde_entropy": None,
            "effectieve_labels": None,
            "duiding": "Geen labelwaarden beschikbaar.",
        }
    probs = counts[counts > 0] / total
    entropy = float(-(probs * np.log(probs)).sum())
    denominator = np.log(max(len(labels), 2))
    normalized = float(entropy / denominator) if denominator else 0.0
    effective_labels = float(np.exp(entropy))
    top_label = str(counts.idxmax())
    top_pct = float(counts.max() / total * 100)
    status = _entropy_status(normalized, top_pct, len(labels))
    if status == "WAARSCHUWING":
        duiding = "Sterke labelconcentratie; controleer mogelijke label collapse of scheve meetbasis."
    elif status == "LET_OP":
        duiding = "Labels zijn duidelijk scheef verdeeld; interpreteer per-label scores voorzichtig."
    else:
        duiding = "Geen sterke aanwijzing voor label collapse op basis van deze verdeling."
    return {
        "test": naam,
        "status": status,
        "n": total,
        "aantal_labels": len(labels),
        "top_label": top_label,
        "top_pct": round(top_pct, 1),
        "shannon_entropy": round(entropy, 3),
        "genormaliseerde_entropy": round(normalized, 3),
        "effectieve_labels": round(effective_labels, 2),
        "duiding": duiding,
    }


def status_exact(old_value: Any, current_value: Any) -> str:
    if current_value is None:
        return "NIET_BEREKEND"
    return "OK" if old_value == current_value else "AANPASSEN"


def status_float(old_value: float, current_value: Any, tolerance: float = 0.01) -> str:
    if current_value is None:
        return "NIET_BEREKEND"
    try:
        return "OK" if abs(float(old_value) - float(current_value)) <= tolerance else "AANPASSEN"
    except Exception:
        return "AANPASSEN"


def nested_get(data: dict[str, Any], keys: list[str], default: Any = None) -> Any:
    value: Any = data
    for key in keys:
        if not isinstance(value, dict) or key not in value:
            return default
        value = value[key]
    return value


def plot_bar(table: pd.DataFrame, x: str, y: str, title: str, color: str = "#4C78A8") -> None:
    if table is None or table.empty or x not in table.columns or y not in table.columns:
        display(Markdown(f"Geen data voor grafiek: {title}"))
        return
    ax = table.plot(kind="bar", x=x, y=y, legend=False, color=color)
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("Aantal")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()


def wilson_ci(successes: Any, total: Any, z: float = 1.959963984540054) -> tuple[float, float]:
    try:
        s, n = int(successes), int(total)
    except Exception:
        return (float("nan"), float("nan"))
    if n <= 0:
        return (float("nan"), float("nan"))
    s = max(0, min(s, n)); p_hat = s / n
    denom = 1 + z**2 / n
    centre = p_hat + z**2 / (2 * n)
    margin = z * np.sqrt((p_hat * (1 - p_hat) + z**2 / (4 * n)) / n)
    return ((centre - margin) / denom, (centre + margin) / denom)

def pct(value: Any, digits: int = 1) -> str:
    try:
        v = float(value)
    except Exception:
        return "n.b."
    return "n.b." if np.isnan(v) else f"{v * 100:.{digits}f}%"

def ci_pct(ci: tuple[float, float], digits: int = 1) -> str:
    lo, hi = ci
    return "n.b." if np.isnan(lo) or np.isnan(hi) else f"{pct(lo, digits)}-{pct(hi, digits)}"

def proportion_record(maat: str, successes: Any, total: Any, digits: int = 1) -> dict[str, Any]:
    try:
        s, n = int(successes), int(total)
        value = s / n if n else float("nan")
    except Exception:
        s, n, value = None, None, float("nan")
    return {"maat": maat, "teller": s, "noemer": n, "waarde": pct(value, digits), "wilson_95bi": ci_pct(wilson_ci(s, n), digits)}

def bootstrap_mean_ci(values: Any, n_bootstrap: int = 1000, seed: int = 20260604, max_items: int = 1000) -> tuple[float, float]:
    vals = pd.to_numeric(pd.Series(values), errors="coerce").dropna().to_numpy(dtype=float)
    if len(vals) == 0 or len(vals) > max_items:
        return (float("nan"), float("nan"))
    rng = np.random.default_rng(seed)
    means = [float(np.mean(rng.choice(vals, size=len(vals), replace=True))) for _ in range(n_bootstrap)]
    return (float(np.quantile(means, 0.025)), float(np.quantile(means, 0.975)))

def binary_f1_score(y_true: Any, y_pred: Any, positive_label: Any) -> float:
    truth, pred, label = pd.Series(y_true).astype(str), pd.Series(y_pred).astype(str), str(positive_label)
    tp = int(((truth == label) & (pred == label)).sum())
    fp = int(((truth != label) & (pred == label)).sum())
    fn = int(((truth == label) & (pred != label)).sum())
    denom = 2 * tp + fp + fn
    return float("nan") if denom == 0 else 2 * tp / denom

def bootstrap_f1_ci(y_true: Any, y_pred: Any, positive_label: Any, n_bootstrap: int = 1000, seed: int = 20260604, max_items: int = 1000) -> tuple[float, float]:
    pairs = pd.DataFrame({"truth": pd.Series(y_true).astype(str), "pred": pd.Series(y_pred).astype(str)}).dropna()
    if pairs.empty or len(pairs) > max_items:
        return (float("nan"), float("nan"))
    rng = np.random.default_rng(seed); scores = []
    for _ in range(n_bootstrap):
        sample = pairs.iloc[rng.integers(0, len(pairs), size=len(pairs))]
        score = binary_f1_score(sample["truth"], sample["pred"], positive_label)
        if not np.isnan(score): scores.append(score)
    return (float(np.quantile(scores, 0.025)), float(np.quantile(scores, 0.975))) if scores else (float("nan"), float("nan"))

In [ ]:
input_rows = [
    dataset_input_row("projectroot", PROJECT_ROOT, "map", "werkruimte"),
    dataset_input_row("thesistekst", THESIS_TEXT_PATH, "tekst", "actuele scriptiebron voor claim-checks"),
    dataset_input_row("thesis index", THESIS_INDEX_PATH, "tekst", "oude validatie-index / referentie"),
    dataset_input_row("DV1 outputmap", DV1_OUTPUT_DIR, "map", "corpusafbakening deelvraag 1"),
    dataset_input_row("DV1 college tellingen", DV1_COLLEGE_DEFINITIES_CSV, "csv", "oude DV1-contextbundel; canonieke scope 171"),
    dataset_input_row("DV1 analysepopulaties", DV1_ANALYSEPOPULATIES_CSV, "csv", "corpus versus analysebasis"),
    dataset_input_row("oude advies summary", OLD_ADVIES_SUMMARY_JSON, "json", "historische adviesvalidatie-output"),
    dataset_input_row("oude advies report", OLD_ADVIES_REPORT_MD, "markdown", "historisch adviesvalidatierapport"),
    dataset_input_row("kabinetsreactie analysebatch", KABINET_BATCH_DIR, "batchmap", "actuele volledige batch voor herberekening"),
    dataset_input_row("kabinetsreactie historische 39-batch", KABINET_HISTORISCHE_VALIDATIE_BATCH_DIR, "batchmap", "oude validatiesample / referentie"),
    dataset_input_row("oude kabinet summary", OLD_KABINET_SUMMARY_JSON, "json", "historische validatiesamenvatting"),
    dataset_input_row("oude kabinet statistiek", OLD_KABINET_STATS_JSON, "json", "historische validatiestatistiek"),
    dataset_input_row("oude kabinet rapport", OLD_KABINET_REPORT_MD, "markdown", "historisch rapport"),
]
for naam, pad in RELIABILITY_JSONS.items():
    input_rows.append(dataset_input_row(f"reliability {naam}", pad, "json", "bestaande test-hertest samenvatting"))

inputcheck = pd.DataFrame(input_rows)
databronnen_overzicht = inputcheck.copy()
inputcheck.drop(columns=["result_jsons"], errors="ignore")

In [ ]:
bronmatrix = pd.DataFrame([
    source_registry_row("adviesvalidatie live DB", PROJECT_ROOT / "pg_database", "code/live", "dynamisch", "adviesextractie wordt opnieuw uit PostgreSQL berekend"),
    source_registry_row("classificatie-metadata golden DB", PROJECT_ROOT / "AI agents" / "AI adviescollege documenten - classificatie and metadata" / "scripts" / "golden_class_meta", "code/live", "dynamisch", "golden vergelijking leest live DB via compare_golden_db.py"),
    source_registry_row("DV1 corpuscontext", DV1_BUNDLE_DIR, "artefact", "gepend", "corpuscontext uit DV1-output"),
    source_registry_row("kabinetsreactie analysebatch", KABINET_BATCH_BUNDLE_DIR, "artefact", "gepend", "ruwe batch; hoofdwaarde na 171-filter staat in 02b"),
    source_registry_row("historische kabinetsreactie 39-batch", KABINET_HISTORISCHE_BUNDLE_DIR, "artefact", "gepend", "oude referentiebatch"),
    source_registry_row("adviesextractie golden-set scores", ADVIES_GOLDEN_DIR, "artefact", "gepend", "scorecards en thesis-scores"),
    source_registry_row("kabinetsreactie reliability", RELIABILITY_BUNDLE_DIR, "artefact", "gepend", "test-hertest JSONs en retry-wisselaars"),
    source_registry_row("kabinetsreactie golden set", KABINET_GOLDEN_DIR, "artefact", "gepend", "ruwe golden-set; hoofdwaarde na 171-filter staat in 02b"),
    source_registry_row("oude claim-referenties", OLD_REFERENCES_BUNDLE_DIR, "artefact", "gepend", "oude summaries en rapporten voor claimcheck"),
    source_registry_row("classificatie-metadata reproduceerbaarheid", CLASSMETA_CONSISTENCY_BUNDLE_DIR, "artefact", "gepend", "100 documenten x 5 runs analyse"),
    source_registry_row("adviesextractie hertestdocumenten", ADVIES_RELIABILITY_DIR, "artefact", "gepend", "hertest/retry docs en thesis-validatieteksten"),
])
bronmatrix_info = {
    "status": "OK" if bronmatrix["bestaat"].all() else "AANPASSEN",
    "detail": f"{int(bronmatrix['bestaat'].sum())}/{len(bronmatrix)} bronlocaties gevonden; live DB-bronnen blijven dynamisch, artefacten staan in {DV2_BRONNEN_DIR.name}.",
}
display(bronmatrix)
bronmatrix_info

In [ ]:
advies_data = None
advies_flags = pd.DataFrame()
advies_stats: dict[str, Any] = {}
advies_labels = pd.DataFrame()
advies_error = None
advies_bron_detail = ""


def _load_advies_from_frozen(bundle_dir: Path):
    from pg_database.checks.advies_validatie.contract import ValidationData

    df_docs = pd.read_csv(bundle_dir / "df_docs.csv")
    df_aanb = pd.read_csv(bundle_dir / "df_aanbevelingen.csv")
    df_prob = pd.read_csv(bundle_dir / "df_problemen.csv")

    def _parse_json_cell(val):
        if isinstance(val, str):
            return json.loads(val)
        if val is None or (np.isscalar(val) and pd.isna(val)):
            return []
        return val

    for col in ["onderzoeksmethoden", "consultatie_vormen", "consultatie_actoren",
                "actor_rollen", "actor_namen", "consultaties_json", "beleidsopties_json",
                "betrokken_actoren_json"]:
        if col in df_docs.columns:
            df_docs[col] = df_docs[col].apply(_parse_json_cell)
    if "box_ids" in df_aanb.columns:
        df_aanb["box_ids"] = df_aanb["box_ids"].apply(_parse_json_cell)
    if "box_ids" in df_prob.columns:
        df_prob["box_ids"] = df_prob["box_ids"].apply(_parse_json_cell)

    for col in ["jaar", "aanbevelingen_aantal", "probleemdefinities_aantal", "meta_aanbevelingen_aantal"]:
        if col in df_docs.columns:
            df_docs[col] = pd.to_numeric(df_docs[col], errors="coerce").astype("Int64")

    from pg_database.checks.advies_validatie.contract import ValidationData as _VD
    return _VD(
        df_docs=df_docs,
        df_aanbevelingen=df_aanb,
        df_problemen=df_prob,
        n_documenten=len(df_docs),
        include_english=INCLUDE_ENGLISH,
        notities=[f"Geladen uit bevroren bundel {bundle_dir.name} (peildatum 2026-06-05)"],
        geladen_op=datetime.now().isoformat(timespec="seconds"),
    )


if USE_FROZEN_ADVIES or RUN_ADVIES_LIVE:
    try:
        from pg_database.checks.advies_validatie import checks_logical as advies_checks
        from pg_database.checks.advies_validatie import data_loading as advies_data_loading
        from pg_database.checks.advies_validatie import reporting as advies_reporting
        from pg_database.checks.advies_validatie import statistics as advies_statistics

        if USE_FROZEN_ADVIES:
            advies_data = _load_advies_from_frozen(ADVIESVALIDATIE_FROZEN_DIR)
            advies_bron_detail = f"bevroren bundel {ADVIESVALIDATIE_FROZEN_DIR.name}"
        else:
            advies_data = advies_data_loading.load_validation_data(include_english=INCLUDE_ENGLISH, scope="corpus_keuzes")
            advies_bron_detail = "live DB (scope=corpus_keuzes, 171)"
        advies_flags = advies_checks.run_logical_checks(advies_data)
        advies_stats = advies_statistics.compute_statistics(advies_data)
        advies_labels = advies_reporting.assign_usability_labels(advies_data, advies_flags)
    except Exception as exc:
        advies_error = repr(exc)

if advies_error:
    display(Markdown(f"**Adviesvalidatie niet berekend. Fout:** `{advies_error}`"))
else:
    display(Markdown(
        f"**Adviesvalidatie berekend uit {advies_bron_detail}.** "
        f"Documenten: {len(advies_data.df_docs)}; aanbevelingen: {len(advies_data.df_aanbevelingen)}; "
        f"probleemdefinities: {len(advies_data.df_problemen)}."
    ))
    display(label_counts(advies_labels))

In [ ]:
advies_summary = {
    "n_documenten": int(len(advies_labels)) if not advies_labels.empty else None,
    "n_groen": int((advies_labels["label"] == "groen").sum()) if "label" in advies_labels else None,
    "n_oranje": int((advies_labels["label"] == "oranje").sum()) if "label" in advies_labels else None,
    "n_rood": int((advies_labels["label"] == "rood").sum()) if "label" in advies_labels else None,
    "n_flags_totaal": int(len(advies_flags)) if advies_flags is not None else None,
    "n_flags_hard": int((advies_flags["ernst"] == "hard").sum()) if "ernst" in advies_flags else None,
    "n_flags_soft": int((advies_flags["ernst"] == "soft").sum()) if "ernst" in advies_flags else None,
    "n_aanbevelingen_rows": int(len(advies_data.df_aanbevelingen)) if advies_data is not None else None,
    "n_problemen_rows": int(len(advies_data.df_problemen)) if advies_data is not None else None,
    "geladen_op": getattr(advies_data, "geladen_op", None) if advies_data is not None else None,
    "include_english": INCLUDE_ENGLISH,
}

display(pd.DataFrame([advies_summary]).T.rename(columns={0: "waarde"}))

if not advies_flags.empty:
    display(Markdown("### Top checks adviesvalidatie"))
    display(value_counts_table(advies_flags, "check_naam", "check_naam").head(10))

if advies_data is not None:
    kernvelden = [
        "aanbevelingen_aantal",
        "probleemdefinities_aantal",
        "n_consultaties",
        "n_unieke_actoren",
        "n_woorden",
    ]
    beschikbare_kernvelden = [veld for veld in kernvelden if veld in advies_data.df_docs.columns]
    display(Markdown("### Advies kernvariabelen"))
    display(advies_data.df_docs[beschikbare_kernvelden].describe().T.round(2))

In [ ]:
plot_bar(label_counts(advies_labels), "label", "aantal", "Adviesvalidatie: groen/oranje/rood", "#59A14F")

if not advies_flags.empty:
    flags_ernst = value_counts_table(advies_flags, "ernst", "ernst")
    plot_bar(flags_ernst, "ernst", "aantal", "Adviesvalidatie: flags per ernst", "#E15759")

if advies_data is not None:
    for veld in ["aanbevelingen_aantal", "probleemdefinities_aantal", "n_consultaties", "n_unieke_actoren"]:
        if veld in advies_data.df_docs.columns:
            advies_data.df_docs[veld].dropna().plot(kind="hist", bins=30, title=f"Advies: verdeling {veld}")
            plt.xlabel(veld)
            plt.tight_layout()
            plt.show()

In [ ]:
result_json_count = count_result_jsons(KABINET_BATCH_DIR)
historical_result_json_count = count_result_jsons(KABINET_HISTORISCHE_VALIDATIE_BATCH_DIR)
kabinet_batch_ok = KABINET_BATCH_DIR.exists() and result_json_count > 0
batch_is_expected_full = result_json_count >= EXPECTED_KABINET_RESULT_JSONS
small_batch_blocked = kabinet_batch_ok and not ALLOW_SMALL_KABINET_BATCH and not batch_is_expected_full

batchcheck = pd.DataFrame([
    {"controle": "actuele batch_dir bestaat", "waarde": KABINET_BATCH_DIR.exists(), "detail": str(KABINET_BATCH_DIR)},
    {"controle": "actuele cases map bestaat", "waarde": (KABINET_BATCH_DIR / "cases").exists(), "detail": str(KABINET_BATCH_DIR / "cases")},
    {"controle": "actuele batch_summary bestaat", "waarde": (KABINET_BATCH_DIR / "batch_summary.json").exists(), "detail": str(KABINET_BATCH_DIR / "batch_summary.json")},
    {"controle": "ruwe batch compleet", "waarde": batch_is_expected_full, "detail": "Bronbatch wordt alleen gebruikt na 171-filter; hoofdwaarde staat in de kabinetcel."},
    {"controle": "actuele batch is filterbaar", "waarde": batch_is_expected_full, "detail": "False betekent: niet gebruiken voor scriptiecijfers"},
    {"controle": "kleine/smoke batch geblokkeerd", "waarde": small_batch_blocked, "detail": "True betekent: herberekening wordt overgeslagen"},
    {"controle": "historische 39-batch aanwezig", "waarde": KABINET_HISTORISCHE_VALIDATIE_BATCH_DIR.exists(), "detail": str(KABINET_HISTORISCHE_VALIDATIE_BATCH_DIR)},
    {"controle": "historische 39-batch result_jsons", "waarde": historical_result_json_count, "detail": f"referentie verwacht {EXPECTED_HISTORISCHE_KABINET_RESULT_JSONS}"},
])
batchcheck

In [ ]:
kabinet_data = None
kabinet_flags = pd.DataFrame()
kabinet_stats: dict[str, Any] = {}
kabinet_labels = pd.DataFrame()
kabinet_error = None
kabinet_bron_detail = ""
kabinet_in_scope_cases: set | None = None

if small_batch_blocked:
    kabinet_error = "Kleine/smoke batch geblokkeerd. Pas config bewust aan als dit toch de doelbatch is."
elif not kabinet_batch_ok:
    kabinet_error = "Geen geldige kabinetsreactie batch gevonden."
else:
    try:
        from pg_database.checks.kabinetsreactie_validatie import checks_logical as kabinet_checks
        from pg_database.checks.kabinetsreactie_validatie import data_loading as kabinet_data_loading
        from pg_database.checks.kabinetsreactie_validatie import reporting as kabinet_reporting
        from pg_database.checks.kabinetsreactie_validatie import statistics as kabinet_statistics

        kabinet_data = kabinet_data_loading.load_validation_data(KABINET_BATCH_DIR)

        if USE_171_SCOPE_KABINET:
            in_scope_df = pd.read_csv(KABINET_171_FROZEN_DIR / "in_scope_documenten.csv", dtype=str)
            kabinet_in_scope_cases = set(in_scope_df["case_id"].astype(str))
            kabinet_data.df_docs = kabinet_data.df_docs[
                kabinet_data.df_docs["case_id"].astype(str).isin(kabinet_in_scope_cases)
            ].reset_index(drop=True)
            kabinet_data.df_items = kabinet_data.df_items[
                kabinet_data.df_items["case_id"].astype(str).isin(kabinet_in_scope_cases)
            ].reset_index(drop=True)
            kabinet_data.n_documenten = len(kabinet_data.df_docs)
            kabinet_bron_detail = (
                f"171-scope (bevroren {KABINET_171_FROZEN_DIR.name}): "
                f"{kabinet_data.n_documenten} docs / {len(kabinet_data.df_items)} elementen"
            )
        else:
            kabinet_bron_detail = (
                f"volledige batch {KABINET_BATCH_DIR.name}: "
                f"{len(kabinet_data.df_docs)} docs / {len(kabinet_data.df_items)} elementen"
            )

        kabinet_flags = kabinet_checks.run_logical_checks(kabinet_data)
        kabinet_stats = kabinet_statistics.compute_statistics(kabinet_data)
        kabinet_labels = kabinet_reporting.assign_usability_labels(kabinet_data, kabinet_flags)
    except Exception as exc:
        kabinet_error = repr(exc)

if kabinet_error:
    display(Markdown(f"**Kabinetsreactievalidatie niet berekend. Reden:** `{kabinet_error}`"))
else:
    display(Markdown(f"**Kabinetsreactievalidatie berekend — {kabinet_bron_detail}.**"))
    display(label_counts(kabinet_labels))

In [ ]:
kabinet_summary = {
    "n_documenten": int(len(kabinet_labels)) if not kabinet_labels.empty else None,
    "n_items": int(len(kabinet_data.df_items)) if kabinet_data is not None else None,
    "n_groen": int((kabinet_labels["label"] == "groen").sum()) if "label" in kabinet_labels else None,
    "n_oranje": int((kabinet_labels["label"] == "oranje").sum()) if "label" in kabinet_labels else None,
    "n_rood": int((kabinet_labels["label"] == "rood").sum()) if "label" in kabinet_labels else None,
    "n_flags_totaal": int(len(kabinet_flags)) if kabinet_flags is not None else None,
    "n_flags_hard": int((kabinet_flags["ernst"] == "hard").sum()) if "ernst" in kabinet_flags else None,
    "n_flags_soft": int((kabinet_flags["ernst"] == "soft").sum()) if "ernst" in kabinet_flags else None,
    "n_flags_info": int((kabinet_flags["ernst"] == "info").sum()) if "ernst" in kabinet_flags else None,
    "geladen_op": getattr(kabinet_data, "geladen_op", None) if kabinet_data is not None else None,
}

if kabinet_data is not None and "advies_element_type" in kabinet_data.df_items.columns:
    type_counts_live = kabinet_data.df_items["advies_element_type"].value_counts().to_dict()
    kabinet_summary["n_aanbeveling"] = int(type_counts_live.get("aanbeveling", 0))
    kabinet_summary["n_probleemdefinitie"] = int(type_counts_live.get("probleemdefinitie", 0))
else:
    type_counts_live = {}
    kabinet_summary["n_aanbeveling"] = None
    kabinet_summary["n_probleemdefinitie"] = None

if kabinet_data is not None and "n_trace_warnings" in kabinet_data.df_items.columns:
    kabinet_summary["items_met_3plus_trace_warnings"] = int((kabinet_data.df_items["n_trace_warnings"] >= 3).sum())
    kabinet_summary["pct_items_met_3plus_trace_warnings"] = round(kabinet_summary["items_met_3plus_trace_warnings"] / max(len(kabinet_data.df_items), 1) * 100, 1)
else:
    kabinet_summary["items_met_3plus_trace_warnings"] = None
    kabinet_summary["pct_items_met_3plus_trace_warnings"] = None

display(pd.DataFrame([kabinet_summary]).T.rename(columns={0: "waarde"}))

if kabinet_data is not None:
    display(Markdown("### Finale verwerkingslabels"))
    display(value_counts_table(kabinet_data.df_items, "finale_verwerkingslabel", "finale_verwerkingslabel"))
    display(Markdown("### Advieselementtypes"))
    display(value_counts_table(kabinet_data.df_items, "advies_element_type", "advies_element_type"))
    display(Markdown("### Kruistabel type x label"))
    display(pd.crosstab(kabinet_data.df_items["advies_element_type"], kabinet_data.df_items["finale_verwerkingslabel"]))

In [ ]:
plot_bar(label_counts(kabinet_labels), "label", "aantal", "Kabinetsreactievalidatie: groen/oranje/rood", "#F28E2B")

if kabinet_data is not None:
    label_table = value_counts_table(kabinet_data.df_items, "finale_verwerkingslabel", "label")
    plot_bar(label_table, "label", "aantal", "Finale verwerkingslabels", "#4E79A7")

    if {"advies_element_type", "finale_verwerkingslabel"}.issubset(kabinet_data.df_items.columns):
        cross = pd.crosstab(kabinet_data.df_items["advies_element_type"], kabinet_data.df_items["finale_verwerkingslabel"])
        ax = cross.T.plot(kind="bar", stacked=True)
        ax.set_title("Finale labels per advieselementtype")
        ax.set_xlabel("finale_verwerkingslabel")
        ax.set_ylabel("Aantal")
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.show()

    if {"cov_ratio_probleemdefinitie", "onduidelijk_ratio_probleemdefinitie"}.issubset(kabinet_data.df_docs.columns):
        ax = kabinet_data.df_docs.plot.scatter(
            x="cov_ratio_probleemdefinitie",
            y="onduidelijk_ratio_probleemdefinitie",
            title="Coverage probleemdefinitie versus onduidelijkheid",
            color="#E15759",
        )
        ax.set_xlabel("coverage ratio probleemdefinitie")
        ax.set_ylabel("onduidelijk ratio probleemdefinitie")
        plt.tight_layout()
        plt.show()

    if "n_trace_warnings" in kabinet_data.df_items.columns:
        trace_table = pd.DataFrame({
            "categorie": ["<3 trace warnings", ">=3 trace warnings"],
            "aantal": [
                int((kabinet_data.df_items["n_trace_warnings"] < 3).sum()),
                int((kabinet_data.df_items["n_trace_warnings"] >= 3).sum()),
            ],
        })
        plot_bar(trace_table, "categorie", "aantal", "Trace warnings per item", "#B07AA1")

In [ ]:
s5_boxcontrole_rows = [
    {"agent": "agent_1", "case_id": "5137__5042", "document_id": "5042", "advies_element_id": "PD-11792", "label": "overgenomen", "n_trace_warnings": 3, "oordeel": "SUPPORTS", "toelichting": "Box 15 p3 zegt dat nieuwe vaccinatie risicogroepen beschermt tegen ziekte/ziekenhuisopname; dat past bij overname."},
    {"agent": "agent_1", "case_id": "5543__5535", "document_id": "5535", "advies_element_id": "AANB-24143", "label": "overgenomen", "n_trace_warnings": 19, "oordeel": "SUPPORTS", "toelichting": "Boxen tonen AstraZeneca-inzet en expliciet 'volg ik het advies'; de boxtekst draagt het label."},
    {"agent": "agent_1", "case_id": "6526__6519", "document_id": "6519", "advies_element_id": "PD-18072", "label": "gedeeltelijk_overgenomen", "n_trace_warnings": 6, "oordeel": "SUPPORTS", "toelichting": "Boxen tonen concrete hulp bij digitale middelen en bestaand beleid; dit ondersteunt gedeeltelijke overname."},
    {"agent": "agent_1", "case_id": "6434__10996", "document_id": "10996", "advies_element_id": "AANB-17848", "label": "gedeeltelijk_overgenomen", "n_trace_warnings": 49, "oordeel": "PARTIAL", "toelichting": "Boxen tonen vooral adviesweergave, kosten en geen middelen beschikbaar; beperking is zichtbaar, positieve overnamekant minder."},
    {"agent": "agent_1", "case_id": "6267__6211", "document_id": "6211", "advies_element_id": "AANB-17734", "label": "herformuleerd", "n_trace_warnings": 3, "oordeel": "PARTIAL", "toelichting": "Box reageert inhoudelijk met steun/alternatief beleid, maar zegt ook 'neem ik niet over'; herformulering is mogelijk maar zwak."},
    {"agent": "agent_1", "case_id": "9376__9355", "document_id": "9355", "advies_element_id": "AANB-12466", "label": "herformuleerd", "n_trace_warnings": 15, "oordeel": "SUPPORTS", "toelichting": "Boxen tonen erkenning, herformulering naar participatieve begroting en procedurele opvolging."},
    {"agent": "agent_2", "case_id": "6854__6840", "document_id": "6840", "advies_element_id": "PD-9090", "label": "gekoppeld_aan_bestaand_beleid", "n_trace_warnings": 3, "oordeel": "PARTIAL", "toelichting": "Passage erkent burgerschap en verwijst naar bestaande wettelijke basis, maar raakt niet alle onderdelen duidelijk."},
    {"agent": "agent_2", "case_id": "8200__8982", "document_id": "8982", "advies_element_id": "AANB-10624", "label": "gekoppeld_aan_bestaand_beleid", "n_trace_warnings": 16, "oordeel": "SUPPORTS", "toelichting": "Passage verwijst naar eerdere brieven, steunpunt en lopende regelingen; bestaand beleid ondersteunt het label."},
    {"agent": "agent_2", "case_id": "7170__10948", "document_id": "10948", "advies_element_id": "PD-9415", "label": "erkend_zonder_actie", "n_trace_warnings": 3, "oordeel": "SUPPORTS", "toelichting": "Passage erkent prijsdruk door de markt, maar noemt geen nieuwe actie."},
    {"agent": "agent_2", "case_id": "8737__8806", "document_id": "8806", "advies_element_id": "AANB-11481", "label": "erkend_zonder_actie", "n_trace_warnings": 15, "oordeel": "PARTIAL", "toelichting": "Passage erkent punten, maar bevat ook bestaande/voorgenomen acties en afwijzing; niet zuiver erkend zonder actie."},
    {"agent": "agent_2", "case_id": "8151__8685", "document_id": "8685", "advies_element_id": "PD-20009", "label": "afgewezen", "n_trace_warnings": 3, "oordeel": "SUPPORTS", "toelichting": "Passage zegt dat fiscale behandeling van de eigen woning niet wijzigt; dit ondersteunt afwijzing."},
    {"agent": "agent_2", "case_id": "2579__2566", "document_id": "2566", "advies_element_id": "AANB-18899", "label": "afgewezen", "n_trace_warnings": 19, "oordeel": "SUPPORTS", "toelichting": "Box zegt expliciet dat het advies geen aanleiding geeft voor extra aandacht; afwijzing is goed ondersteund."},
    {"agent": "agent_3", "case_id": "10865__10866", "document_id": "10866", "advies_element_id": "AANB-26774", "label": "niet_herkenbaar_verwerkt", "n_trace_warnings": 3, "oordeel": "SUPPORTS", "toelichting": "Box beschrijft huidig afstammingsrecht en erkenning, maar geen terminologiewijziging naar verwantschapsrecht."},
    {"agent": "agent_3", "case_id": "9588__9561", "document_id": "9561", "advies_element_id": "AANB-5343", "label": "niet_herkenbaar_verwerkt", "n_trace_warnings": 3, "oordeel": "SUPPORTS", "toelichting": "Box vat samenwerking en verschillen samen, maar bevat geen kabinetstandpunt over bestuurscultuurverschillen."},
    {"agent": "agent_3", "case_id": "2690__2657", "document_id": "2657", "advies_element_id": "PD-13054", "label": "onduidelijk", "n_trace_warnings": 4, "oordeel": "CANNOT_ASSESS", "toelichting": "Geen concrete kabinetsreactie_passages/box_ids beschikbaar; inhoudelijke broncontrole niet mogelijk."},
    {"agent": "agent_3", "case_id": "10838__10839", "document_id": "10839", "advies_element_id": "AANB-5544", "label": "onduidelijk", "n_trace_warnings": 28, "oordeel": "SUPPORTS", "toelichting": "Boxen tonen meerdere relevante maar gemengde maatregelen; onduidelijk is verdedigbaar."},
    {"agent": "agent_3", "case_id": "10096__11121", "document_id": "11121", "advies_element_id": "AANB-6637", "label": "gedeeltelijk_overgenomen", "n_trace_warnings": 22, "oordeel": "SUPPORTS", "toelichting": "Boxen tonen instemming met meer onderhandelmacht, maar ook bezwaren tegen instrumenten; gedeeltelijke overname past."},
    {"agent": "agent_3", "case_id": "6880__6810", "document_id": "6810", "advies_element_id": "AANB-13438", "label": "onduidelijk", "n_trace_warnings": 22, "oordeel": "SUPPORTS", "toelichting": "Boxen bevatten samenvatting, afwijzing en andere beleidsacties; signalen zijn gemengd."},
]

s5_boxcontrole = pd.DataFrame(s5_boxcontrole_rows)
s5_boxcontrole_summary = (
    s5_boxcontrole["oordeel"]
    .value_counts()
    .rename_axis("oordeel")
    .reset_index(name="aantal")
)
s5_boxcontrole_summary["percentage"] = (s5_boxcontrole_summary["aantal"] / len(s5_boxcontrole) * 100).round(1)
order = {"SUPPORTS": 0, "PARTIAL": 1, "CANNOT_ASSESS": 2, "NOT_SUPPORT": 3}
s5_boxcontrole_summary["_order"] = s5_boxcontrole_summary["oordeel"].map(order).fillna(99)
s5_boxcontrole_summary = s5_boxcontrole_summary.sort_values("_order").drop(columns="_order")

display(Markdown("**S5-boxcontrole: samenvatting van 18 handmatig beoordeelde items**"))
display(s5_boxcontrole_summary)
display(Markdown(
    "**Conclusie:** in deze steekproef ondersteunt de box/source passage het AI-label bij "
    f"{int((s5_boxcontrole['oordeel'] == 'SUPPORTS').sum())}/{len(s5_boxcontrole)} items duidelijk, "
    f"bij {int((s5_boxcontrole['oordeel'] == 'PARTIAL').sum())} items gedeeltelijk, "
    f"en bij {int((s5_boxcontrole['oordeel'] == 'CANNOT_ASSESS').sum())} item was broncontrole niet mogelijk. "
    "Er was geen item waarbij de box/source passage het label duidelijk niet ondersteunde. "
    "S5 moet daarom vooral worden gelezen als citaat- en traceerbaarheidsruis, niet als directe inhoudelijke fout."
))
display(s5_boxcontrole[["case_id", "advies_element_id", "label", "n_trace_warnings", "oordeel", "toelichting"]])

In [ ]:
robustness_records: list[dict[str, Any]] = []


def missing_columns(df: pd.DataFrame | None, columns: list[str]) -> list[str]:
    if df is None or df.empty:
        return list(columns)
    return [column for column in columns if column not in df.columns]


def add_robustness_record(
    test: str,
    status: str,
    detail: str,
    missing: list[str] | None = None,
    onderdeel: str = "algemeen",
) -> None:
    robustness_records.append({
        "test": test,
        "onderdeel": onderdeel,
        "status": status,
        "detail": detail,
        "ontbrekende_kolommen": ", ".join(missing or []),
    })


def prepare_advies_docs_labeled() -> pd.DataFrame:
    if advies_data is None or advies_labels is None or advies_labels.empty:
        add_robustness_record("advies join", "NIET_UITVOERBAAR", "advies_data of advies_labels ontbreekt", onderdeel="advies")
        return pd.DataFrame()
    required_docs = ["unique_id"]
    required_labels = ["unique_id", "label"]
    missing = missing_columns(advies_data.df_docs, required_docs) + missing_columns(advies_labels, required_labels)
    if missing:
        add_robustness_record("advies join", "NIET_UITVOERBAAR", "kan adviesdata niet aan labels koppelen", missing, "advies")
        return pd.DataFrame()
    docs = advies_data.df_docs.copy()
    labels = advies_labels[["unique_id", "label", "n_hard", "n_soft", "redenen"]].copy()
    return docs.merge(labels, on="unique_id", how="left")


def prepare_kabinet_docs_labeled() -> pd.DataFrame:
    if kabinet_data is None or kabinet_labels is None or kabinet_labels.empty:
        add_robustness_record("kabinet join", "NIET_UITVOERBAAR", "kabinet_data of kabinet_labels ontbreekt", onderdeel="kabinet")
        return pd.DataFrame()
    required_docs = ["case_id", "document_id"]
    required_labels = ["case_id", "label"]
    missing = missing_columns(kabinet_data.df_docs, required_docs) + missing_columns(kabinet_labels, required_labels)
    if missing:
        add_robustness_record("kabinet join", "NIET_UITVOERBAAR", "kan kabinetsdata niet aan labels koppelen", missing, "kabinet")
        return pd.DataFrame()
    docs = kabinet_data.df_docs.copy()
    label_cols = [column for column in ["case_id", "label", "n_hard", "n_soft", "n_info", "redenen"] if column in kabinet_labels.columns]
    return docs.merge(kabinet_labels[label_cols], on="case_id", how="left")


def attach_flag_counts(df: pd.DataFrame, flags: pd.DataFrame, key: str) -> pd.DataFrame:
    if df is None or df.empty or flags is None or flags.empty or key not in df.columns or key not in flags.columns or "ernst" not in flags.columns:
        return df.copy() if df is not None else pd.DataFrame()
    counts = flags.groupby([key, "ernst"]).size().unstack(fill_value=0).reset_index()
    rename_map = {column: f"flags_{column}" for column in counts.columns if column != key}
    counts = counts.rename(columns=rename_map)
    merged = df.merge(counts, on=key, how="left")
    for column in ["flags_hard", "flags_soft", "flags_info"]:
        if column not in merged.columns:
            merged[column] = 0
        merged[column] = merged[column].fillna(0).astype(int)
    merged["flags_totaal"] = merged[["flags_hard", "flags_soft", "flags_info"]].sum(axis=1)
    return merged


def available_columns(df: pd.DataFrame, columns: list[str]) -> list[str]:
    if df is None or df.empty:
        return []
    return [column for column in columns if column in df.columns]


def numeric_by_group(df: pd.DataFrame, group_col: str, numeric_cols: list[str]) -> pd.DataFrame:
    cols = available_columns(df, numeric_cols)
    if df is None or df.empty or group_col not in df.columns or not cols:
        return pd.DataFrame()
    parts = []
    for column in cols:
        summary = df.groupby(group_col)[column].agg(["count", "mean", "median", "std"]).reset_index()
        summary.insert(1, "variabele", column)
        parts.append(summary)
    return pd.concat(parts, ignore_index=True).round(3)


def compare_green_vs_usable(df: pd.DataFrame, numeric_cols: list[str], onderdeel: str) -> pd.DataFrame:
    cols = available_columns(df, numeric_cols)
    if df is None or df.empty or "label" not in df.columns or not cols:
        add_robustness_record("groen_vs_groen_oranje", "NIET_UITVOERBAAR", "label of numerieke kolommen ontbreken", ["label"] + numeric_cols, onderdeel)
        return pd.DataFrame()
    green = df[df["label"] == "groen"]
    usable = df[df["label"].isin(["groen", "oranje"])]
    if green.empty or usable.empty:
        add_robustness_record("groen_vs_groen_oranje", "NIET_UITVOERBAAR", "groene of bruikbare set is leeg", onderdeel=onderdeel)
        return pd.DataFrame()
    rows = []
    for column in cols:
        rows.append({
            "onderdeel": onderdeel,
            "variabele": column,
            "n_groen": int(green[column].notna().sum()),
            "mediaan_groen": green[column].median(),
            "n_groen_oranje": int(usable[column].notna().sum()),
            "mediaan_groen_oranje": usable[column].median(),
            "verschil_mediaan": usable[column].median() - green[column].median(),
        })
    add_robustness_record("groen_vs_groen_oranje", "OK", f"{len(cols)} variabelen vergeleken", onderdeel=onderdeel)
    return pd.DataFrame(rows).round(3)


def make_period(value: Any) -> str:
    if pd.isna(value):
        return "onbekend"
    try:
        year = int(value)
    except Exception:
        return "onbekend"
    if year < 2005:
        return "voor_2005"
    if year <= 2009:
        return "2005-2009"
    if year <= 2014:
        return "2010-2014"
    if year <= 2019:
        return "2015-2019"
    if year <= 2024:
        return "2020-2024"
    return "na_2024"


def iqr_outlier_mask(series: pd.Series) -> pd.Series:
    clean = pd.to_numeric(series, errors="coerce")
    q1 = clean.quantile(0.25)
    q3 = clean.quantile(0.75)
    iqr = q3 - q1
    if pd.isna(iqr) or iqr == 0:
        return pd.Series(False, index=series.index)
    return (clean < q1 - 1.5 * iqr) | (clean > q3 + 1.5 * iqr)


advies_docs_labeled = attach_flag_counts(prepare_advies_docs_labeled(), advies_flags, "unique_id")
kabinet_docs_labeled = attach_flag_counts(prepare_kabinet_docs_labeled(), kabinet_flags, "document_id")

preflight_rows = [
    {"onderdeel": "advies", "object": "advies_docs_labeled", "rijen": len(advies_docs_labeled), "kolommen": len(advies_docs_labeled.columns) if not advies_docs_labeled.empty else 0},
    {"onderdeel": "kabinet", "object": "kabinet_docs_labeled", "rijen": len(kabinet_docs_labeled), "kolommen": len(kabinet_docs_labeled.columns) if not kabinet_docs_labeled.empty else 0},
]
preflight_table = pd.DataFrame(preflight_rows)
preflight_table

In [ ]:
advies_numeric_core = [
    "n_woorden",
    "aanbevelingen_aantal",
    "probleemdefinities_aantal",
    "n_consultaties",
    "n_unieke_actoren",
    "n_beleidsopties",
    "n_betrokken_actoren",
]
kabinet_numeric_core = [
    "advies_elementen_totaal",
    "candidate_coverage_ratio",
    "cov_ratio_aanbeveling",
    "cov_ratio_probleemdefinitie",
    "onduidelijk_ratio",
    "onduidelijk_ratio_aanbeveling",
    "onduidelijk_ratio_probleemdefinitie",
    "n_reviewpunten",
    "n_gestopte_candidate_pairs",
]

advies_green_compare = compare_green_vs_usable(advies_docs_labeled, advies_numeric_core, "advies")
kabinet_green_compare = compare_green_vs_usable(kabinet_docs_labeled, kabinet_numeric_core, "kabinet")
green_compare = pd.concat([advies_green_compare, kabinet_green_compare], ignore_index=True)
display(green_compare)

if not advies_docs_labeled.empty and "label" in advies_docs_labeled.columns:
    plot_bar(label_counts(advies_docs_labeled), "label", "aantal", "Advies: selectie groen/oranje/rood", "#59A14F")
if not kabinet_docs_labeled.empty and "label" in kabinet_docs_labeled.columns:
    plot_bar(label_counts(kabinet_docs_labeled), "label", "aantal", "Kabinet: selectie groen/oranje/rood", "#F28E2B")

for df, cols, title in [
    (advies_docs_labeled, ["n_consultaties", "n_unieke_actoren", "n_woorden"], "Advies kernvariabelen per label"),
    (kabinet_docs_labeled, ["candidate_coverage_ratio", "onduidelijk_ratio"], "Kabinet kwaliteit per label"),
]:
    available = available_columns(df, cols)
    if df is not None and not df.empty and "label" in df.columns and available:
        df.boxplot(column=available, by="label", figsize=(10, 4), rot=30)
        plt.suptitle("")
        plt.title(title)
        plt.tight_layout()
        plt.show()

In [ ]:
quality_tables = []

if not advies_docs_labeled.empty and "label" in advies_docs_labeled.columns:
    advies_quality = numeric_by_group(
        advies_docs_labeled,
        "label",
        ["n_woorden", "aanbevelingen_aantal", "probleemdefinities_aantal", "n_consultaties", "n_unieke_actoren", "flags_totaal", "flags_hard", "flags_soft"],
    )
    advies_quality.insert(0, "onderdeel", "advies") if not advies_quality.empty else None
    quality_tables.append(advies_quality)
    add_robustness_record("kwaliteitsbias", "OK", "advieslabels en flags samengevat", onderdeel="advies")
else:
    add_robustness_record("kwaliteitsbias", "NIET_UITVOERBAAR", "advies labeldata ontbreekt", ["label"], "advies")

if not kabinet_docs_labeled.empty and "label" in kabinet_docs_labeled.columns:
    kabinet_quality = numeric_by_group(
        kabinet_docs_labeled,
        "label",
        ["candidate_coverage_ratio", "onduidelijk_ratio", "n_reviewpunten", "n_gestopte_candidate_pairs", "flags_totaal", "flags_hard", "flags_soft", "flags_info"],
    )
    kabinet_quality.insert(0, "onderdeel", "kabinet") if not kabinet_quality.empty else None
    quality_tables.append(kabinet_quality)
    add_robustness_record("kwaliteitsbias", "OK", "kabinetlabels en flags samengevat", onderdeel="kabinet")
else:
    add_robustness_record("kwaliteitsbias", "NIET_UITVOERBAAR", "kabinet labeldata ontbreekt", ["label"], "kabinet")

quality_bias_table = pd.concat([table for table in quality_tables if table is not None and not table.empty], ignore_index=True) if quality_tables else pd.DataFrame()
display(quality_bias_table)

if not advies_flags.empty and {"ernst", "check_naam"}.issubset(advies_flags.columns):
    display(Markdown("### Advies: top flags"))
    display(value_counts_table(advies_flags, "check_naam", "check_naam").head(10))
if not kabinet_flags.empty and {"ernst", "check_naam"}.issubset(kabinet_flags.columns):
    display(Markdown("### Kabinet: top flags"))
    display(value_counts_table(kabinet_flags, "check_naam", "check_naam").head(10))

for df, column, title in [
    (advies_docs_labeled, "flags_totaal", "Advies: flags totaal per label"),
    (kabinet_docs_labeled, "flags_totaal", "Kabinet: flags totaal per label"),
]:
    if df is not None and not df.empty and {"label", column}.issubset(df.columns):
        df.boxplot(column=column, by="label", figsize=(8, 4), rot=30)
        plt.suptitle("")
        plt.title(title)
        plt.tight_layout()
        plt.show()

In [ ]:
period_tables = []

if not advies_docs_labeled.empty and {"jaar", "label"}.issubset(advies_docs_labeled.columns):
    advies_period = advies_docs_labeled.copy()
    advies_period["periode"] = advies_period["jaar"].apply(make_period)
    advies_period_table = pd.crosstab(advies_period["periode"], advies_period["label"], margins=True).reset_index()
    period_tables.append(advies_period_table.assign(onderdeel="advies"))
    add_robustness_record("periode_effect", "OK", "adviesperiode op jaar berekend", onderdeel="advies")
    display(Markdown("### Advies: labelverdeling per periode"))
    display(advies_period_table)
    plot_data = advies_period[advies_period["periode"] != "onbekend"].groupby(["periode", "label"]).size().unstack(fill_value=0)
    if not plot_data.empty:
        plot_data.plot(kind="bar", stacked=True, title="Advies: labelverdeling per periode")
        plt.xlabel("periode")
        plt.ylabel("Aantal")
        plt.xticks(rotation=30, ha="right")
        plt.tight_layout()
        plt.show()
else:
    add_robustness_record("periode_effect", "NIET_UITVOERBAAR", "adviesjaar of label ontbreekt", ["jaar", "label"], "advies")

if not kabinet_docs_labeled.empty and {"jaar", "label"}.issubset(kabinet_docs_labeled.columns):
    kabinet_period = kabinet_docs_labeled.copy()
    kabinet_period["periode"] = kabinet_period["jaar"].apply(make_period)
    kabinet_period_table = pd.crosstab(kabinet_period["periode"], kabinet_period["label"], margins=True).reset_index()
    period_tables.append(kabinet_period_table.assign(onderdeel="kabinet"))
    add_robustness_record("periode_effect", "OK", "kabinetperiode op jaar berekend", onderdeel="kabinet")
    display(Markdown("### Kabinet: labelverdeling per periode"))
    display(kabinet_period_table)
else:
    add_robustness_record("periode_effect", "NIET_UITVOERBAAR", "kabinetjaar ontbreekt in contract", ["jaar"], "kabinet")

In [ ]:
group_effect_tables = []

if not advies_docs_labeled.empty and {"college", "label"}.issubset(advies_docs_labeled.columns):
    college_summary = advies_docs_labeled.groupby("college").agg(
        n=("unique_id", "count"),
        pct_groen=("label", lambda s: round((s == "groen").mean() * 100, 1)),
        pct_oranje=("label", lambda s: round((s == "oranje").mean() * 100, 1)),
        mediaan_woorden=("n_woorden", "median") if "n_woorden" in advies_docs_labeled.columns else ("label", "count"),
    ).reset_index()
    college_summary = college_summary.sort_values("n", ascending=False)
    group_effect_tables.append(college_summary.assign(onderdeel="advies_college"))
    add_robustness_record("college_effect", "OK", "advies per college samengevat", onderdeel="advies")
    display(college_summary.head(15))
    top_colleges = college_summary.head(15)["college"].tolist()
    plot_data = advies_docs_labeled[advies_docs_labeled["college"].isin(top_colleges)].groupby(["college", "label"]).size().unstack(fill_value=0)
    if not plot_data.empty:
        plot_data.plot(kind="bar", stacked=True, figsize=(11, 5), title="Advies: labels voor top-15 colleges")
        plt.xlabel("college")
        plt.ylabel("Aantal")
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.show()
else:
    add_robustness_record("college_effect", "NIET_UITVOERBAAR", "college of label ontbreekt", ["college", "label"], "advies")

optional_group_cols = ["type", "type_college", "college_type", "domein", "beleidsdomein"]
for group_col in optional_group_cols:
    if not advies_docs_labeled.empty and {group_col, "label"}.issubset(advies_docs_labeled.columns):
        table = pd.crosstab(advies_docs_labeled[group_col], advies_docs_labeled["label"], margins=True).reset_index()
        display(Markdown(f"### Advies optioneel: {group_col}"))
        display(table)
        add_robustness_record("type_domein_effect", "OK", f"{group_col} aanwezig", onderdeel="advies")
        break
else:
    add_robustness_record("type_domein_effect", "NIET_UITVOERBAAR", "geen type/domein-kolom in vaste runtime-data", optional_group_cols, "advies")

for group_col in ["college", "type", "type_college", "college_type", "domein", "beleidsdomein"]:
    if not kabinet_docs_labeled.empty and {group_col, "label"}.issubset(kabinet_docs_labeled.columns):
        table = pd.crosstab(kabinet_docs_labeled[group_col], kabinet_docs_labeled["label"], margins=True).reset_index()
        display(Markdown(f"### Kabinet optioneel: {group_col}"))
        display(table)
        add_robustness_record("kabinet_groep_effect", "OK", f"{group_col} aanwezig", onderdeel="kabinet")
        break
else:
    add_robustness_record("kabinet_groep_effect", "NIET_UITVOERBAAR", "geen groepkolom in kabinetcontract", ["college", "type", "domein"], "kabinet")

In [ ]:
interactivity_required = ["n_consultaties", "n_unieke_actoren", "veldconsultatie_niveau"]
missing_interactivity = missing_columns(advies_docs_labeled, interactivity_required)

if missing_interactivity or advies_docs_labeled.empty:
    add_robustness_record("interactiviteit_proxy", "NIET_UITVOERBAAR", "interactiviteitskolommen ontbreken", missing_interactivity, "advies")
    interactivity_table = pd.DataFrame()
else:
    proxy_df = advies_docs_labeled.copy()
    niveau_map = {"geen": 0, "beperkt": 1, "hoog": 2, "onbekend": np.nan, "": np.nan}
    proxy_df["veldconsultatie_ordinaal"] = proxy_df["veldconsultatie_niveau"].map(niveau_map)
    corr_rows = []
    for left, right in [
        ("n_consultaties", "veldconsultatie_ordinaal"),
        ("n_unieke_actoren", "veldconsultatie_ordinaal"),
        ("n_consultaties", "n_unieke_actoren"),
    ]:
        valid = proxy_df[[left, right]].dropna()
        corr_rows.append({
            "paar": f"{left} vs {right}",
            "spearman_rho": valid[left].corr(valid[right], method="spearman") if len(valid) >= 3 else np.nan,
            "n": len(valid),
        })
    interactivity_table = pd.DataFrame(corr_rows).round(3)
    add_robustness_record("interactiviteit_proxy", "OK", "proxies onderling vergeleken", onderdeel="advies")
    display(interactivity_table)

    for proxy_col in ["n_consultaties", "n_unieke_actoren"]:
        try:
            proxy_df[f"{proxy_col}_kwartiel"] = pd.qcut(proxy_df[proxy_col].rank(method="first"), 4, labels=["Q1 laag", "Q2", "Q3", "Q4 hoog"])
            table = pd.crosstab(proxy_df[f"{proxy_col}_kwartiel"], proxy_df["label"], margins=True).reset_index()
            display(Markdown(f"### Advies: labelverdeling per kwartiel {proxy_col}"))
            display(table)
        except Exception as exc:
            add_robustness_record("interactiviteit_kwartielen", "WAARSCHUWING", f"kwartielen niet berekend voor {proxy_col}: {exc}", onderdeel="advies")

    if {"n_consultaties", "veldconsultatie_ordinaal"}.issubset(proxy_df.columns):
        ax = proxy_df.plot.scatter(x="n_consultaties", y="veldconsultatie_ordinaal", title="Samenhang consultaties en veldconsultatie-niveau")
        ax.set_ylabel("veldconsultatie_niveau ordinaal")
        plt.tight_layout()
        plt.show()

In [ ]:
outlier_rows = []
advies_outlier_cols = available_columns(advies_docs_labeled, ["n_woorden", "aanbevelingen_aantal", "probleemdefinities_aantal", "n_consultaties", "n_unieke_actoren"])

if not advies_docs_labeled.empty and advies_outlier_cols:
    outlier_mask = pd.Series(False, index=advies_docs_labeled.index)
    for column in advies_outlier_cols:
        mask = iqr_outlier_mask(advies_docs_labeled[column])
        outlier_mask = outlier_mask | mask
        outlier_rows.append({
            "onderdeel": "advies",
            "variabele": column,
            "n_outliers": int(mask.sum()),
            "pct_outliers": round(mask.mean() * 100, 1),
        })
    advies_no_outliers = advies_docs_labeled[~outlier_mask].copy()
    sensitivity_rows = []
    for column in advies_outlier_cols:
        sensitivity_rows.append({
            "variabele": column,
            "mediaan_volledig": advies_docs_labeled[column].median(),
            "mediaan_zonder_outliers": advies_no_outliers[column].median() if not advies_no_outliers.empty else np.nan,
            "verschil": (advies_no_outliers[column].median() - advies_docs_labeled[column].median()) if not advies_no_outliers.empty else np.nan,
        })
    outlier_table = pd.DataFrame(outlier_rows)
    outlier_sensitivity_table = pd.DataFrame(sensitivity_rows).round(3)
    add_robustness_record("outlier_sensitiviteit", "OK", f"{len(advies_outlier_cols)} adviesvariabelen getest", onderdeel="advies")
    display(Markdown("### Advies: outliers per variabele"))
    display(outlier_table)
    display(Markdown("### Advies: mediaan volledig versus zonder outliers"))
    display(outlier_sensitivity_table)
else:
    outlier_table = pd.DataFrame()
    outlier_sensitivity_table = pd.DataFrame()
    add_robustness_record("outlier_sensitiviteit", "NIET_UITVOERBAAR", "advies-outlierkolommen ontbreken", ["n_woorden", "aanbevelingen_aantal", "probleemdefinities_aantal"], "advies")

kabinet_outlier_cols = available_columns(kabinet_docs_labeled, ["candidate_coverage_ratio", "onduidelijk_ratio", "n_reviewpunten", "n_gestopte_candidate_pairs"])
if not kabinet_docs_labeled.empty and kabinet_outlier_cols:
    status = "WAARSCHUWING" if len(kabinet_docs_labeled) < 50 else "OK"
    detail = "kleine kabinetset; interpreteer als gevoeligheidscheck" if status == "WAARSCHUWING" else "kabinet-outliers getest"
    add_robustness_record("outlier_sensitiviteit", status, detail, onderdeel="kabinet")
    kabinet_outlier_summary = []
    for column in kabinet_outlier_cols:
        mask = iqr_outlier_mask(kabinet_docs_labeled[column])
        kabinet_outlier_summary.append({
            "onderdeel": "kabinet",
            "variabele": column,
            "n_outliers": int(mask.sum()),
            "pct_outliers": round(mask.mean() * 100, 1),
        })
    display(Markdown("### Kabinet: outliers per variabele"))
    display(pd.DataFrame(kabinet_outlier_summary))
else:
    add_robustness_record("outlier_sensitiviteit", "NIET_UITVOERBAAR", "kabinet-outlierkolommen ontbreken", ["candidate_coverage_ratio", "onduidelijk_ratio"], "kabinet")

In [ ]:
robustness_summary = pd.DataFrame(robustness_records)
status_order = {"OK": 0, "WAARSCHUWING": 1, "NIET_UITVOERBAAR": 2}
if not robustness_summary.empty:
    robustness_summary["_order"] = robustness_summary["status"].map(status_order).fillna(99)
    robustness_summary = robustness_summary.sort_values(["_order", "onderdeel", "test"]).drop(columns="_order")
robustness_summary

In [ ]:
old_kabinet_stats = read_json(OLD_KABINET_STATS_JSON)
old_kabinet_report_text = read_text(OLD_KABINET_REPORT_MD)

def parse_first_int(text: str, pattern: str) -> int | None:
    match = re.search(pattern, text, flags=re.IGNORECASE)
    if not match:
        return None
    return int(match.group(1).replace(".", ""))


report_aanbevelingen = parse_first_int(old_kabinet_report_text, r"([0-9.]+)\s+aanbevelingen\s*\+")
report_probleemdef = parse_first_int(old_kabinet_report_text, r"\+\s*([0-9.]+)\s+probleemdefinities")
report_overgenomen = parse_first_int(old_kabinet_report_text, r"\|\s*overgenomen\s*\|\s*([0-9.]+)\s*\|")
report_afgewezen = parse_first_int(old_kabinet_report_text, r"\|\s*afgewezen\s*\|\s*([0-9.]+)\s*\|")
report_onduidelijk = parse_first_int(old_kabinet_report_text, r"\|\s*onduidelijk\s*\|\s*([0-9.]+)\s*\|")

old_json_type_counts = nested_get(old_kabinet_stats, ["entropie", "advies_element_type", "top3"], {}) or {}
old_json_label_counts = nested_get(old_kabinet_stats, ["label_verdeling", "totaal"], {}) or {}

conflict_rows = [
    {
        "metric": "aantal aanbevelingen",
        "rapport": report_aanbevelingen,
        "oude_json": old_json_type_counts.get("aanbeveling"),
        "live": kabinet_summary.get("n_aanbeveling"),
    },
    {
        "metric": "aantal probleemdefinities",
        "rapport": report_probleemdef,
        "oude_json": old_json_type_counts.get("probleemdefinitie"),
        "live": kabinet_summary.get("n_probleemdefinitie"),
    },
    {
        "metric": "label overgenomen",
        "rapport": report_overgenomen,
        "oude_json": old_json_label_counts.get("overgenomen"),
        "live": None if kabinet_data is None else int((kabinet_data.df_items["finale_verwerkingslabel"] == "overgenomen").sum()),
    },
    {
        "metric": "label afgewezen",
        "rapport": report_afgewezen,
        "oude_json": old_json_label_counts.get("afgewezen"),
        "live": None if kabinet_data is None else int((kabinet_data.df_items["finale_verwerkingslabel"] == "afgewezen").sum()),
    },
    {
        "metric": "label onduidelijk",
        "rapport": report_onduidelijk,
        "oude_json": old_json_label_counts.get("onduidelijk"),
        "live": None if kabinet_data is None else int((kabinet_data.df_items["finale_verwerkingslabel"] == "onduidelijk").sum()),
    },
]
bronconflict = pd.DataFrame(conflict_rows)
bronconflict["status"] = np.where(bronconflict["rapport"] == bronconflict["oude_json"], "OK", "BRONCONFLICT")
bronconflict["actie"] = np.where(
    bronconflict["status"] == "BRONCONFLICT",
    "Gebruik live notebookcijfer of machine-JSON; oude rapporttekst niet overnemen.",
    "Geen conflict tussen rapport en JSON.",
)
bronconflict

In [ ]:
reliability_rows = []
for experiment, path in RELIABILITY_JSONS.items():
    data = read_json(path)
    metadata = data.get("metadata", {}) if isinstance(data, dict) else {}
    aggregate = data.get("aggregate", {}) if isinstance(data, dict) else {}
    reliability_rows.append({
        "experiment": experiment,
        "bron": str(path),
        "bestaat": path.exists(),
        "n_cases": metadata.get("n_cases"),
        "n_runs_per_case": metadata.get("n_runs_per_case"),
        "mode": metadata.get("mode"),
        "workers": metadata.get("workers"),
        "mean_percentage_agreement": aggregate.get("mean_percentage_agreement"),
        "mean_krippendorff_alpha": aggregate.get("mean_krippendorff_alpha"),
        "cases_failed": aggregate.get("cases_failed"),
        "rerun_gedaan": False,
    })

reliability_table = pd.DataFrame(reliability_rows)
reliability_table

In [ ]:
KABINET_GOLDEN_SET_CSV = KABINET_GOLDEN_DIR / "golden_set.csv"
KABINET_GOLDEN_VS_RUN_CSV = KABINET_GOLDEN_DIR / "golden_vs_run.csv"
KABINET_GOLDEN_EXPECTED_ROWS = 88

kabinet_golden_uitval_cases: set = set()
if USE_171_SCOPE_KABINET:
    _uitval_csv = KABINET_171_FROZEN_DIR / "uitval_documenten.csv"
    if _uitval_csv.exists():
        kabinet_golden_uitval_cases = set(pd.read_csv(_uitval_csv, dtype=str)["case_id"].astype(str))


def _filter_golden_in_scope(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty or "case" not in df.columns or not kabinet_golden_uitval_cases:
        return df
    return df[~df["case"].astype(str).isin(kabinet_golden_uitval_cases)].reset_index(drop=True)


kabinetsreactie_golden_set_bestanden = pd.DataFrame([
    {"bestand": "golden_set.csv", "rol": "duurzame golden set", "pad": str(KABINET_GOLDEN_SET_CSV), "bestaat": KABINET_GOLDEN_SET_CSV.exists(), "rijen": count_csv_data_rows(KABINET_GOLDEN_SET_CSV), "verwachte_rijen": KABINET_GOLDEN_EXPECTED_ROWS},
    {"bestand": "golden_vs_run.csv", "rol": "golden set versus agent-run", "pad": str(KABINET_GOLDEN_VS_RUN_CSV), "bestaat": KABINET_GOLDEN_VS_RUN_CSV.exists(), "rijen": count_csv_data_rows(KABINET_GOLDEN_VS_RUN_CSV), "verwachte_rijen": KABINET_GOLDEN_EXPECTED_ROWS},
])
kabinetsreactie_golden_set_bestanden["rijen_ok"] = kabinetsreactie_golden_set_bestanden["rijen"] == kabinetsreactie_golden_set_bestanden["verwachte_rijen"]
golden_bestanden_ok = kabinetsreactie_golden_set_bestanden["bestaat"].all() and kabinetsreactie_golden_set_bestanden["rijen_ok"].all()

golden_reference_reliability = _filter_golden_in_scope(pd.read_csv(KABINET_GOLDEN_SET_CSV)) if KABINET_GOLDEN_SET_CSV.exists() else pd.DataFrame()
consensus_counts = golden_reference_reliability["consensus"].value_counts().to_dict() if "consensus" in golden_reference_reliability.columns else {}
meerderheid_counts = golden_reference_reliability["meerderheid"].value_counts().to_dict() if "meerderheid" in golden_reference_reliability.columns else {}

golden_item_count = int(len(golden_reference_reliability))
golden_unaniem = int(consensus_counts.get("unaniem", 0))
golden_meerderheid = int(consensus_counts.get("meerderheid", 0))
golden_splits = int(consensus_counts.get("split", 0) + consensus_counts.get("splits", 0))
golden_ambigu_meerderheid = int(meerderheid_counts.get("ambigu", 0))

_scope_detail = f" (171-scope: {len(kabinet_golden_uitval_cases)} case(s) buiten scope)" if kabinet_golden_uitval_cases else ""
kabinetsreactie_golden_set_info = {
    "status": "OK" if golden_bestanden_ok else "AANPASSEN",
    "detail": f"{golden_item_count} golden-set items in scope{_scope_detail}; {golden_unaniem} unaniem, {golden_meerderheid} meerderheid, {golden_splits} splits; {golden_ambigu_meerderheid} ambigu-meerderheid.",
    "n_items": golden_item_count,
    "n_unaniem": golden_unaniem,
    "n_meerderheid": golden_meerderheid,
    "n_splits": golden_splits,
    "ambigu_meerderheid": golden_ambigu_meerderheid,
}

display(Markdown(f"**Conclusie:** {kabinetsreactie_golden_set_info['detail']}"))
display(kabinetsreactie_golden_set_bestanden.drop(columns=["rijen", "verwachte_rijen"], errors="ignore"))
kabinetsreactie_golden_set_info

In [ ]:
kabinetsreactie_golden_label_eval = pd.DataFrame()
kabinetsreactie_confusion_matrix = pd.DataFrame()
kabinetsreactie_confusion_row_pct = pd.DataFrame()
kabinetsreactie_label_metrics = pd.DataFrame()
kabinetsreactie_accuracy_ci = pd.DataFrame()
kabinetsreactie_confusion_info = {
    "status": "NIET_BEREKEND",
    "detail": "Golden-set vergelijking niet berekend.",
}

if KABINET_GOLDEN_SET_CSV.exists() and KABINET_GOLDEN_VS_RUN_CSV.exists():
    golden_reference = _filter_golden_in_scope(pd.read_csv(KABINET_GOLDEN_SET_CSV))
    golden_vs_run = _filter_golden_in_scope(pd.read_csv(KABINET_GOLDEN_VS_RUN_CSV))
    expected_golden_rows = len(golden_reference)
    kabinetsreactie_golden_label_eval = golden_vs_run.merge(
        golden_reference[["case", "advies_element_id", "type", "consensus"]],
        on=["case", "advies_element_id"],
        how="left",
        validate="one_to_one",
    )
    kabinetsreactie_golden_label_eval["coarse_klopt_bool"] = kabinetsreactie_golden_label_eval["coarse_klopt"].astype(str).str.lower().eq("ja")

    label_order = ["substantieel", "procedureel", "retorisch", "afgewezen", "niet_behandeld", "ambigu"]
    observed_labels = sorted(
        set(kabinetsreactie_golden_label_eval["coarse_truth"].dropna().astype(str))
        | set(kabinetsreactie_golden_label_eval["ai_coarse"].dropna().astype(str))
    )
    ordered_labels = [label for label in label_order if label in observed_labels] + [label for label in observed_labels if label not in label_order]

    kabinetsreactie_confusion_matrix = pd.crosstab(
        kabinetsreactie_golden_label_eval["coarse_truth"],
        kabinetsreactie_golden_label_eval["ai_coarse"],
        dropna=False,
    ).reindex(index=ordered_labels, columns=ordered_labels, fill_value=0)
    kabinetsreactie_confusion_row_pct = (
        kabinetsreactie_confusion_matrix.div(kabinetsreactie_confusion_matrix.sum(axis=1).replace(0, np.nan), axis=0) * 100
    ).round(1).fillna(0)

    metric_rows = []
    for label in ordered_labels:
        support = int((kabinetsreactie_golden_label_eval["coarse_truth"] == label).sum())
        predicted = int((kabinetsreactie_golden_label_eval["ai_coarse"] == label).sum())
        correct = int(((kabinetsreactie_golden_label_eval["coarse_truth"] == label) & (kabinetsreactie_golden_label_eval["ai_coarse"] == label)).sum())
        wrong_predictions = (
            kabinetsreactie_golden_label_eval.loc[
                (kabinetsreactie_golden_label_eval["coarse_truth"] == label)
                & (kabinetsreactie_golden_label_eval["ai_coarse"] != label),
                "ai_coarse",
            ].fillna("<leeg>").value_counts()
        )
        f1_value = binary_f1_score(kabinetsreactie_golden_label_eval["coarse_truth"], kabinetsreactie_golden_label_eval["ai_coarse"], label)
        f1_ci = bootstrap_f1_ci(kabinetsreactie_golden_label_eval["coarse_truth"], kabinetsreactie_golden_label_eval["ai_coarse"], label)
        metric_rows.append({
            "label": label,
            "support_truth": support,
            "predicted_as_label": predicted,
            "correct": correct,
            "recall_pct": round(correct / support * 100, 1) if support else None,
            "recall_95bi": ci_pct(wilson_ci(correct, support)),
            "precision_pct": round(correct / predicted * 100, 1) if predicted else None,
            "precision_95bi": ci_pct(wilson_ci(correct, predicted)),
            "f1": round(f1_value, 3) if not np.isnan(f1_value) else None,
            "f1_bootstrap_95bi": ci_pct(f1_ci),
            "grootste_verwarring": None if wrong_predictions.empty else f"{wrong_predictions.index[0]} ({int(wrong_predictions.iloc[0])})",
        })
    kabinetsreactie_label_metrics = pd.DataFrame(metric_rows)

    total_items = int(len(kabinetsreactie_golden_label_eval))
    total_correct = int(kabinetsreactie_golden_label_eval["coarse_klopt_bool"].sum())
    accuracy_pct = round(total_correct / max(total_items, 1) * 100, 1)
    accuracy_interval = wilson_ci(total_correct, total_items)
    kabinetsreactie_accuracy_ci = pd.DataFrame([proportion_record("accuracy grove labels", total_correct, total_items)])
    wrong_pairs = (
        kabinetsreactie_golden_label_eval.loc[~kabinetsreactie_golden_label_eval["coarse_klopt_bool"]]
        .assign(verward_met=lambda df: df["coarse_truth"].astype(str) + " -> " + df["ai_coarse"].astype(str))["verward_met"]
        .value_counts()
    )
    top_confusion = "geen" if wrong_pairs.empty else f"{wrong_pairs.index[0]} ({int(wrong_pairs.iloc[0])})"
    kabinetsreactie_confusion_info = {
        "status": "OK" if total_items == expected_golden_rows else "AANPASSEN",
        "detail": f"{total_correct}/{total_items} grove labels correct ({accuracy_pct}%; 95%-BI {ci_pct(accuracy_interval)}); grootste verwarring: {top_confusion}.",
        "accuracy_pct": accuracy_pct,
        "accuracy_95bi": ci_pct(accuracy_interval),
        "top_confusion": top_confusion,
    }

    display(Markdown(f"**Samenvatting:** {kabinetsreactie_confusion_info['detail']}"))
    display(Markdown("### Accuracy met Wilson-95%-BI"))
    display(kabinetsreactie_accuracy_ci)
    display(Markdown("### Confusion matrix: aantallen"))
    display(kabinetsreactie_confusion_matrix)
    display(Markdown("### Confusion matrix: rijpercentages"))
    display(kabinetsreactie_confusion_row_pct)
    display(Markdown("### Per-label precision/recall/F1 met 95%-BI"))
    display(kabinetsreactie_label_metrics)
else:
    display(Markdown("**NIET_BEREKEND:** golden-set CSV of golden-vs-run CSV ontbreekt."))

kabinetsreactie_confusion_info

In [ ]:
kabinetsreactie_nonrandom_tables = {}
kabinetsreactie_nonrandom_info = {
    "status": "NIET_BEREKEND",
    "detail": "Non-random dekking/missingness niet berekend.",
}

def grouped_error_table(df: pd.DataFrame, group_col: str) -> pd.DataFrame:
    if df is None or df.empty or group_col not in df.columns or "coarse_klopt_bool" not in df.columns:
        return pd.DataFrame(columns=[group_col, "n", "fouten", "fout_pct"])
    grouped = (
        df.groupby(group_col, dropna=False)
        .agg(
            n=("advies_element_id", "size"),
            fouten=("coarse_klopt_bool", lambda values: int((~values).sum())),
        )
        .reset_index()
    )
    grouped["fout_pct"] = (grouped["fouten"] / grouped["n"].replace(0, np.nan) * 100).round(1).fillna(0)
    return grouped.sort_values(["fout_pct", "n"], ascending=[False, False])

def grouped_trace_table(df: pd.DataFrame, group_col: str) -> pd.DataFrame:
    if df is None or df.empty or group_col not in df.columns or "n_trace_warnings" not in df.columns:
        return pd.DataFrame(columns=[group_col, "n", "trace_3plus", "trace_3plus_pct", "gem_trace_warnings"])
    tmp = df.copy()
    tmp["trace_3plus"] = tmp["n_trace_warnings"].fillna(0) >= 3
    grouped = (
        tmp.groupby(group_col, dropna=False)
        .agg(
            n=(group_col, "size"),
            trace_3plus=("trace_3plus", "sum"),
            gem_trace_warnings=("n_trace_warnings", "mean"),
        )
        .reset_index()
    )
    grouped["trace_3plus"] = grouped["trace_3plus"].astype(int)
    grouped["trace_3plus_pct"] = (grouped["trace_3plus"] / grouped["n"].replace(0, np.nan) * 100).round(1).fillna(0)
    grouped["gem_trace_warnings"] = grouped["gem_trace_warnings"].round(2)
    return grouped.sort_values(["trace_3plus_pct", "n"], ascending=[False, False])

if not kabinetsreactie_golden_label_eval.empty:
    kabinetsreactie_nonrandom_tables["golden_fouten_per_label"] = grouped_error_table(kabinetsreactie_golden_label_eval, "coarse_truth")
    kabinetsreactie_nonrandom_tables["golden_fouten_per_type"] = grouped_error_table(kabinetsreactie_golden_label_eval, "type")
    kabinetsreactie_nonrandom_tables["golden_fouten_per_consensus"] = grouped_error_table(kabinetsreactie_golden_label_eval, "consensus")

    merge_missing = int(kabinetsreactie_golden_label_eval[["type", "consensus"]].isna().any(axis=1).sum())
    max_label_row = kabinetsreactie_nonrandom_tables["golden_fouten_per_label"].head(1)
    max_label_detail = "geen labelverschil" if max_label_row.empty else f"hoogste golden-foutpct bij {max_label_row.iloc[0]['coarse_truth']} ({max_label_row.iloc[0]['fout_pct']}%, n={int(max_label_row.iloc[0]['n'])})"

    live_trace_detail = "trace-warningverdeling niet berekend"
    if kabinet_data is not None and "n_trace_warnings" in kabinet_data.df_items.columns:
        if "finale_verwerkingslabel" in kabinet_data.df_items.columns:
            kabinetsreactie_nonrandom_tables["live_trace_per_label"] = grouped_trace_table(kabinet_data.df_items, "finale_verwerkingslabel")
            top_trace = kabinetsreactie_nonrandom_tables["live_trace_per_label"].head(1)
            if not top_trace.empty:
                live_trace_detail = f"hoogste trace>=3 bij {top_trace.iloc[0]['finale_verwerkingslabel']} ({top_trace.iloc[0]['trace_3plus_pct']}%, n={int(top_trace.iloc[0]['n'])})"
        if "advies_element_type" in kabinet_data.df_items.columns:
            kabinetsreactie_nonrandom_tables["live_trace_per_type"] = grouped_trace_table(kabinet_data.df_items, "advies_element_type")

    status = "OK" if merge_missing == 0 else "AANPASSEN"
    if live_trace_detail != "trace-warningverdeling niet berekend":
        status = "WAARSCHUWING"
    kabinetsreactie_nonrandom_info = {
        "status": status,
        "detail": f"Golden-set merge-missings: {merge_missing}/{len(kabinetsreactie_golden_label_eval)}; {max_label_detail}; {live_trace_detail}.",
        "merge_missing": merge_missing,
    }

    display(Markdown(f"**Samenvatting:** {kabinetsreactie_nonrandom_info['detail']}"))
    display(Markdown("### Golden-set fouten per handmatig label"))
    display(kabinetsreactie_nonrandom_tables["golden_fouten_per_label"])
    display(Markdown("### Golden-set fouten per itemtype"))
    display(kabinetsreactie_nonrandom_tables["golden_fouten_per_type"])
    display(Markdown("### Golden-set fouten per consensusniveau"))
    display(kabinetsreactie_nonrandom_tables["golden_fouten_per_consensus"])
    if "live_trace_per_label" in kabinetsreactie_nonrandom_tables:
        display(Markdown("### Volledige batch: trace warnings per AI-label"))
        display(kabinetsreactie_nonrandom_tables["live_trace_per_label"])
    if "live_trace_per_type" in kabinetsreactie_nonrandom_tables:
        display(Markdown("### Volledige batch: trace warnings per advieselementtype"))
        display(kabinetsreactie_nonrandom_tables["live_trace_per_type"])
else:
    display(Markdown("**NIET_BEREKEND:** voer eerst de confusion-matrixcel uit."))

kabinetsreactie_nonrandom_info

In [ ]:
RETRY_WISSELAARS_TOP20_CSV = RELIABILITY_BUNDLE_DIR / "retry_wisselaars_top20.csv"
RETRY_WISSELAARS_TOP20_MD = RELIABILITY_BUNDLE_DIR / "retry_wisselaars_top20.md"

kabinetsreactie_retry_wisselaars_info = {"status": "NIET_GEVONDEN", "detail": "retry_wisselaars_top20.csv niet gevonden"}
retry_wisselaars_top20 = pd.DataFrame()
retry_labelwisselparen = pd.DataFrame()

if RETRY_WISSELAARS_TOP20_CSV.exists():
    retry_wisselaars_top20 = pd.read_csv(RETRY_WISSELAARS_TOP20_CSV)
    kabinetsreactie_retry_wisselaars_info["status"] = "OK"
    display(Markdown(f"### Top20 controlelijst ({len(retry_wisselaars_top20)} rijen)"))
    display(retry_wisselaars_top20)

    pair_counts: dict[str, int] = {}
    label_columns = ["run0_label", "run1_label", "run2_label"]
    for _, row in retry_wisselaars_top20.iterrows():
        labels = []
        for column in label_columns:
            label = row.get(column)
            if pd.notna(label) and label not in labels:
                labels.append(str(label))
        for left_index, left_label in enumerate(labels):
            for right_label in labels[left_index + 1:]:
                pair = " <-> ".join(sorted([left_label, right_label]))
                pair_counts[pair] = pair_counts.get(pair, 0) + 1

    retry_labelwisselparen = pd.DataFrame(
        [{"labelwisselpaar": pair, "aantal_in_top20": count} for pair, count in pair_counts.items()]
    )
    if not retry_labelwisselparen.empty:
        retry_labelwisselparen["met_onduidelijk"] = retry_labelwisselparen["labelwisselpaar"].str.contains("onduidelijk", regex=False)
        retry_labelwisselparen = retry_labelwisselparen.sort_values(
            ["aantal_in_top20", "labelwisselpaar"], ascending=[False, True]
        ).reset_index(drop=True)
        display(Markdown("### Samenvatting labelwisselparen in de top20"))
        display(retry_labelwisselparen)

        totaal_paren = int(retry_labelwisselparen["aantal_in_top20"].sum())
        onduidelijk_paren = int(retry_labelwisselparen.loc[retry_labelwisselparen["met_onduidelijk"], "aantal_in_top20"].sum())
        kabinetsreactie_retry_wisselaars_info["detail"] = (
            f"{len(retry_wisselaars_top20)} rijen; {onduidelijk_paren}/{totaal_paren} labelwisselparen bevatten onduidelijk"
        )
        display(Markdown(
            "**Conclusie:** het itemniveau is instabiel. In deze top20 zitten vooral 3-way wissels. "
            "Veel wisselparen bevatten `onduidelijk`, vaak tegenover inhoudelijke labels zoals "
            "`overgenomen`, `herformuleerd` en `gerelativeerd`, of tegenover negatieve labels zoals "
            "`afgewezen` en `niet_herkenbaar_verwerkt`."
        ))
    else:
        kabinetsreactie_retry_wisselaars_info["detail"] = f"{len(retry_wisselaars_top20)} rijen; geen labelwisselparen berekend"
elif RETRY_WISSELAARS_TOP20_MD.exists():
    kabinetsreactie_retry_wisselaars_info = {"status": "OK", "detail": "Alleen markdownsamenvatting gevonden; CSV ontbreekt"}
    display(Markdown(read_text(RETRY_WISSELAARS_TOP20_MD)))
else:
    display(Markdown("**NIET_GEVONDEN:** `retry_wisselaars_top20.csv` is niet gevonden."))

kabinetsreactie_retry_wisselaars_info

In [ ]:
import os

ADVIES_RETRY_SEARCH_ROOTS = [
    ADVIES_RELIABILITY_DIR,
]
ADVIES_RETRY_KEYWORDS = ("retry", "hertest", "reliability", "betrouwbaarheid")
ADVIES_SUMMARY_KEYWORDS = ("resultaten", "summary", "samenvatting", "stability")
ADVIES_RETRY_SUFFIXES = {".md", ".csv", ".json"}
ADVIES_STATIC_VALIDATIE_PATHS = [
    ADVIES_RELIABILITY_DIR / "advies_validatie_historisch.md",
    ADVIES_RELIABILITY_DIR / "thesis_validatie_resultaten.md",
    ADVIES_RELIABILITY_DIR / "thesis_validatiesectie_extractie.md",
]

def project_relative(path: Path) -> str:
    try:
        return path.resolve().relative_to(PROJECT_ROOT).as_posix()
    except ValueError:
        return str(path)


def classify_advies_retry_path(path: Path) -> tuple[str, bool]:
    name = path.name.lower()
    path_text = project_relative(path).lower()
    if "kabinetsreactie" in path_text:
        return "uitgesloten: kabinetsreactie", False
    if name == "test_hertest_resultaten.md" or any(keyword in name for keyword in ADVIES_SUMMARY_KEYWORDS):
        return "bruikbare samenvatting/data", True
    if "plan" in name or "plan" in path_text:
        return "plan/documentatie, geen resultaat", False
    return "documentatie of ruwe data, geen samenvatting", False


def read_json_preview(path: Path) -> dict[str, Any]:
    try:
        return read_json(path)
    except json.JSONDecodeError:
        return json.loads(path.read_text(encoding="utf-8-sig"))


found_retry_paths: dict[str, Path] = {}
missing_roots = []
for search_root in ADVIES_RETRY_SEARCH_ROOTS:
    if not search_root.exists():
        missing_roots.append(project_relative(search_root))
        continue
    for current_root, _, filenames in os.walk(search_root, onerror=lambda error: None):
        current_root_path = Path(current_root)
        current_root_text = project_relative(current_root_path).lower()
        retry_dir = any(keyword in current_root_text for keyword in ADVIES_RETRY_KEYWORDS)
        for filename in filenames:
            filename_lower = filename.lower()
            filename_has_retry = any(keyword in filename_lower for keyword in ADVIES_RETRY_KEYWORDS)
            summary_in_retry_dir = retry_dir and any(keyword in filename_lower for keyword in ADVIES_SUMMARY_KEYWORDS)
            if not filename_has_retry and not summary_in_retry_dir:
                continue
            if filename_lower.endswith("_result.json") or filename_lower.endswith("_input.json"):
                continue
            path = current_root_path / filename
            if path.suffix.lower() not in ADVIES_RETRY_SUFFIXES:
                continue
            if "__pycache__" in path.parts:
                continue
            found_retry_paths[str(path.resolve()).lower()] = path

advies_retry_rows = []
for path in sorted(found_retry_paths.values(), key=lambda item: project_relative(item).lower()):
    classificatie, bruikbaar = classify_advies_retry_path(path)
    if classificatie.startswith("uitgesloten"):
        continue
    advies_retry_rows.append({
        "pad": project_relative(path),
        "type": path.suffix.lower().lstrip("."),
        "classificatie": classificatie,
        "bruikbaar": bruikbaar,
    })

advies_retry_table = pd.DataFrame(advies_retry_rows)
bruikbare_advies_retry = advies_retry_table[advies_retry_table["bruikbaar"]] if not advies_retry_table.empty else pd.DataFrame()
bronbundel_retry_gevonden = (
    not bruikbare_advies_retry.empty
    and bruikbare_advies_retry["pad"].str.startswith("thesis/Analyse/DV2_validatie_bronnen/").any()
)

if not advies_retry_table.empty:
    display(Markdown("### Gevonden adviesextractie retry/test-hertest bestanden"))
    display(advies_retry_table)
else:
    display(Markdown("### Gevonden adviesextractie retry/test-hertest bestanden: geen"))

static_validatie_refs = pd.DataFrame([
    {"pad": project_relative(path), "bestaat": path.exists(), "duiding": "golden/statistische validatie, geen retry-summary"}
    for path in ADVIES_STATIC_VALIDATIE_PATHS
])
display(Markdown("### Bestaande adviesextractie-validatie zonder retry/test-hertest"))
display(static_validatie_refs)

if bruikbare_advies_retry.empty:
    adviesextractie_retry_info = {
        "status": "NIET_GEVONDEN",
        "detail": "Geen bruikbare adviesextractie retry/test-hertest samenvatting gevonden; wel golden/statistische validatiebestanden aanwezig.",
    }
    display(Markdown(
        "**NIET_GEVONDEN:** in de gevonden thesisbestanden staat geen vergelijkbare adviesextractie retry-summary. "
        "Er is wel golden/statistische adviesextractie-validatie aanwezig, maar die meet iets anders dan retry/test-hertest stabiliteit."
    ))
else:
    preferred_path_text = None
    for path_text in bruikbare_advies_retry["pad"].tolist():
        if path_text.endswith("tests/test_hertest_resultaten.md") or path_text.endswith("adviesextractie_test_hertest_resultaten.md"):
            preferred_path_text = path_text
            break
    if preferred_path_text is None:
        preferred_path_text = bruikbare_advies_retry.iloc[0]["pad"]
    preferred_path = PROJECT_ROOT / preferred_path_text
    adviesextractie_retry_info = {
        "status": "OK",
        "detail": f"{len(bruikbare_advies_retry)} bruikbare bestand(en); bronbundel bevat retry-summary: {'ja' if bronbundel_retry_gevonden else 'nee'}; hoofdbron: {preferred_path_text}",
    }

    preview_rows = []
    for path_text in bruikbare_advies_retry["pad"].tolist():
        path = PROJECT_ROOT / path_text
        if path.suffix.lower() == ".md":
            content = read_text(path)
            first_heading = next((line.strip("# ") for line in content.splitlines() if line.startswith("#")), "geen kop gevonden")
            preview_rows.append({"pad": path_text, "type": "md", "samenvatting": f"{len(content)} tekens; eerste kop: {first_heading}"})
        elif path.suffix.lower() == ".csv":
            data = pd.read_csv(path)
            preview_rows.append({"pad": path_text, "type": "csv", "samenvatting": f"{data.shape[0]} rijen, {data.shape[1]} kolommen"})
        elif path.suffix.lower() == ".json":
            data = read_json_preview(path)
            keys = list(data.keys())[:6] if isinstance(data, dict) else []
            preview_rows.append({"pad": path_text, "type": "json", "samenvatting": f"{len(keys)} getoonde top-level keys: {', '.join(keys)}"})
    display(Markdown("### Ingelezen bruikbare adviesextractie retry/test-hertest bestanden"))
    display(pd.DataFrame(preview_rows))

    display(Markdown(f"### Ingelezen hoofdbron: `{preferred_path_text}`"))
    if preferred_path.suffix.lower() == ".md":
        display(Markdown(read_text(preferred_path)))
    elif preferred_path.suffix.lower() == ".csv":
        display(pd.read_csv(preferred_path))
    elif preferred_path.suffix.lower() == ".json":
        display(pd.DataFrame([read_json_preview(preferred_path)]).T.rename(columns={0: "waarde"}))

adviesextractie_retry_info

In [ ]:
GOLDEN_DIR = ADVIES_GOLDEN_DIR
GOLDEN_SCORECARDS = GOLDEN_DIR / "scorecards.csv"
GOLDEN_MANIFEST = GOLDEN_DIR / "match_manifest.json"
GOLDEN_THESIS_SCORES = ADVIES_GOLDEN_DIR / "thesis_validatie_scores.csv"

RUN_GOLDEN_LIVE = False

In [ ]:
golden_dekking_info = {"status": "OK", "detail": ""}
golden_dekking_records = []

try:
    manifest = read_json(GOLDEN_MANIFEST)
    if "_missing" not in manifest:
        total = len(manifest)
        matched = sum(1 for case in manifest.values() if case and case.get('status') == 'matched')
        unmatched_list = [gs for gs, case in manifest.items() if not case or case.get('status') != 'matched']
        
        golden_dekking_info["detail"] = f"{matched}/{total} gekoppeld+gevalideerd ({len(unmatched_list)} unmatched: {', '.join(unmatched_list)})"
        
        for gs in unmatched_list:
            case = manifest.get(gs, {})
            reason = case.get('reason', 'onbekend') if case else 'geen case-data'
            golden_dekking_records.append({"gs": gs, "status": "unmatched", "reason": reason})
    else:
        golden_dekking_info["status"] = "NIET_BEREKEND"
        golden_dekking_info["detail"] = f"Manifest niet gevonden: {manifest.get('_missing')}"
except Exception as e:
    golden_dekking_info["status"] = "FOUT"
    golden_dekking_info["detail"] = str(e)

if golden_dekking_records:
    dekking_df = pd.DataFrame(golden_dekking_records)
    display(Markdown(f"### Unmatched cases ({len(golden_dekking_records)} stuks)"))
    display(dekking_df)
else:
    display(Markdown(f"### Status: {golden_dekking_info['detail']}"))

In [ ]:
golden_scores_info = {"status": "OK", "detail": ""}
golden_scores_data = {}

try:
    if not GOLDEN_SCORECARDS.exists():
        raise FileNotFoundError(str(GOLDEN_SCORECARDS))
    
    scores_df = pd.read_csv(GOLDEN_SCORECARDS)
    golden_scores_info["detail"] = f"{scores_df.shape[0]} gescande rapporten"
    
    kernel_cols = [
        "recommendation_semantic_coverage_f1",
        "recommendation_semantic_item_f1",
        "problem_semantic_coverage_f1",
        "problem_semantic_item_f1",
        "metadata_accuracy",
        "field_completion_rate",
        "report_label_accuracy",
        "policy_logic_link_f1",
        "consultation_actor_f1"
    ]
    
    available_cols = [c for c in kernel_cols if c in scores_df.columns]
    
    score_records = []
    for col in available_cols:
        vals = pd.to_numeric(scores_df[col], errors='coerce').dropna()
        if len(vals) > 0:
            golden_scores_data[col] = {
                'mean': float(vals.mean()),
                'min': float(vals.min()),
                'max': float(vals.max()),
                'n': int(len(vals))
            }
            bootstrap_ci = bootstrap_mean_ci(vals) if col.endswith("_f1") else (float("nan"), float("nan"))
            score_records.append({
                "maat": col.replace('_', '-'),
                "gemiddeld": f"{vals.mean():.4f}",
                "bootstrap_f1_95bi": ci_pct(bootstrap_ci, digits=1) if col.endswith("_f1") else "n.v.t.",
                "min": f"{vals.min():.4f}",
                "max": f"{vals.max():.4f}",
                "n": len(vals)
            })
    
    score_df = pd.DataFrame(score_records)
    display(Markdown("### Kernmaten (semantische e5-matching):"))
    display(score_df.set_index("maat"))
    
    if GOLDEN_THESIS_SCORES.exists():
        try:
            thesis_scores = pd.read_csv(GOLDEN_THESIS_SCORES)
            display(Markdown("### Thesis-validatie samenvattingsscores:"))
            display(thesis_scores)
        except Exception as e:
            pass
    
except Exception as e:
    golden_scores_info["status"] = "FOUT"
    golden_scores_info["detail"] = str(e)

In [ ]:
if golden_scores_data:
    plot_data = pd.DataFrame([
        {"maat": col.replace('_', '-'), "gemiddeld": stats['mean']}
        for col, stats in golden_scores_data.items()
    ])
    plot_bar(plot_data, "maat", "gemiddeld", "Golden-set: gemiddelde scores per maat (e5-semantisch)")

In [ ]:
if golden_scores_data:
    def _mean_for(metric: str) -> float | None:
        stats = golden_scores_data.get(metric)
        return None if not stats else stats.get("mean")

    advies_kern_metrics = [
        "recommendation_semantic_coverage_f1",
        "recommendation_semantic_item_f1",
        "problem_semantic_coverage_f1",
        "problem_semantic_item_f1",
        "metadata_accuracy",
        "field_completion_rate",
    ]
    advies_zwakke_metrics = ["consultation_actor_f1", "report_label_accuracy", "policy_logic_link_f1"]
    kern_values = [value for value in (_mean_for(metric) for metric in advies_kern_metrics) if value is not None]
    kern_range = "n.b." if not kern_values else f"{min(kern_values):.2f}-{max(kern_values):.2f}"
    available_weak = [(metric, _mean_for(metric)) for metric in advies_zwakke_metrics if _mean_for(metric) is not None]
    zwakste_metric = min(available_weak, key=lambda item: item[1]) if available_weak else None
    zwakste_text = "niet berekend" if zwakste_metric is None else f"{zwakste_metric[0].replace('_', '-')} = {zwakste_metric[1]:.2f}"
    display(Markdown(
        "### Samenvatting\n\n"
        f"**Sterk:** de kernmaten voor aanbevelingen, probleemstellingen en metadata liggen samen in de bandbreedte `{kern_range}`.\n\n"
        f"**Voorzichtig gebruiken:** de zwakste gemeten submaat is `{zwakste_text}`. Gebruik deze subvelden daarom niet als harde hoofdclaim zonder extra handmatige controle.\n\n"
        "Detailanalyses staan in de bestanden in `ADVIES_GOLDEN_DIR`; deze cel leest alleen bestaande output."
    ))
else:
    display(Markdown("### Samenvatting\n\n**NIET_BEREKEND:** er zijn geen golden-set scores geladen."))

In [ ]:
import importlib.util

RUN_GOLDEN_CLASSMETA_LIVE = True
CLASSMETA_ROOT = PROJECT_ROOT / "AI agents" / "AI adviescollege documenten - classificatie and metadata"
CLASSMETA_COMPARE_PY = CLASSMETA_ROOT / "scripts" / "golden_class_meta" / "compare_golden_db.py"

golden_classmeta_info = {"status": "OK", "detail": ""}
classmeta_reports: dict[str, Any] = {}

try:
    if not RUN_GOLDEN_CLASSMETA_LIVE:
        raise RuntimeError("RUN_GOLDEN_CLASSMETA_LIVE=False; live herberekening overgeslagen.")
    if not CLASSMETA_COMPARE_PY.exists():
        raise FileNotFoundError(str(CLASSMETA_COMPARE_PY))
    _spec = importlib.util.spec_from_file_location("compare_golden_db", CLASSMETA_COMPARE_PY)
    _cgdb = importlib.util.module_from_spec(_spec)
    _spec.loader.exec_module(_cgdb)
    for _src in ("canonical", "raw"):
        classmeta_reports[_src] = _cgdb.compare_golden_db(
            _cgdb.DEFAULT_GOLDEN_DIR, Path("."), source=_src, write_reports=False
        )
    _can = classmeta_reports["canonical"]["summary"]
    _raw = classmeta_reports["raw"]["summary"]
    golden_classmeta_info["detail"] = (
        f"classificatie {_can['classification_matches']}/{_can['classification_coverage']}; "
        f"metadata-dekking canonical {_can['metadata_coverage']}/{_can['total_golden_cases']} "
        f"(raw {_raw['metadata_coverage']}/{_raw['total_golden_cases']})"
    )
except Exception as e:
    golden_classmeta_info["status"] = "FOUT" if RUN_GOLDEN_CLASSMETA_LIVE else "NIET_BEREKEND"
    golden_classmeta_info["detail"] = str(e)

display(Markdown(f"### Status: {golden_classmeta_info['detail']}"))

In [ ]:
if classmeta_reports:
    can = classmeta_reports["canonical"]["summary"]
    raw = classmeta_reports["raw"]["summary"]
    kern_rows = [
        {"maat": "classificatie-dekking", "canonical": can["classification_coverage"], "raw": raw["classification_coverage"], "van": can["total_golden_cases"], "wilson_95bi": ci_pct(wilson_ci(can["classification_coverage"], can["total_golden_cases"]))},
        {"maat": "classificatie exact-match", "canonical": can["classification_matches"], "raw": raw["classification_matches"], "van": can["classification_coverage"], "wilson_95bi": ci_pct(wilson_ci(can["classification_matches"], can["classification_coverage"]))},
        {"maat": "metadata-dekking", "canonical": can["metadata_coverage"], "raw": raw["metadata_coverage"], "van": can["total_golden_cases"], "wilson_95bi": ci_pct(wilson_ci(can["metadata_coverage"], can["total_golden_cases"]))},
    ]
    display(Markdown("### Kerncijfers (canonical = dashboard-view; raw = alle geproduceerde metadata)"))
    display(pd.DataFrame(kern_rows).set_index("maat"))
    display(Markdown(
        f"- Classificatie exact-match (canonical): **{can['classification_matches']}/{can['classification_coverage']}** "
        f"({can['classification_match_rate_over_covered']:.1%})  \n"
        f"- Vergeleken classificatievelden: {', '.join(can['compared_classification_fields'])}  \n"
        f"- Niet in DB (niet vergeleken): {', '.join(can['classification_fields_not_in_db'])}"
    ))

In [ ]:
if classmeta_reports:
    can = classmeta_reports["canonical"]["summary"]
    cls_rows = [{"veld": f, "match": st["matches"], "n": st["total"], "accuratesse": round(st["rate"], 3), "wilson_95bi": ci_pct(wilson_ci(st["matches"], st["total"]))}
                for f, st in can["classification_field_accuracy"].items()]
    display(Markdown("### Classificatie: accuratesse per veld (canonical)"))
    display(pd.DataFrame(cls_rows).set_index("veld"))

    meta_rows = [{"veld": f, "match": st["matches"], "n": st["total"], "accuratesse": round(st["rate"], 3), "wilson_95bi": ci_pct(wilson_ci(st["matches"], st["total"]))}
                 for f, st in can["metadata_field_accuracy"].items()]
    meta_df = pd.DataFrame(meta_rows)
    display(Markdown("### Metadata: accuratesse per veld (canonical, slechtste eerst)"))
    display(meta_df.set_index("veld"))
    if not meta_df.empty:
        plot_bar(meta_df.assign(pct=(meta_df["accuratesse"] * 100).round(1)), "veld", "pct",
                 "Golden vs DB: metadata-accuratesse per veld (%) - canonical")

In [ ]:
if classmeta_reports:
    can = classmeta_reports["canonical"]["summary"]
    raw = classmeta_reports["raw"]["summary"]
    class_rate = can["classification_match_rate_over_covered"]
    metadata_coverage_rate = can["metadata_coverage"] / can["total_golden_cases"] if can["total_golden_cases"] else float("nan")
    display(Markdown(
        "### Samenvatting\n\n"
        f"- **Classificatie:** `{can['classification_matches']}/{can['classification_coverage']}` exact-match over de vergeleken velden "
        f"(`{pct(class_rate)}`, Wilson-95%-BI `{ci_pct(wilson_ci(can['classification_matches'], can['classification_coverage']))}`).\n"
        f"- **Metadata:** canonical dekking `{can['metadata_coverage']}/{can['total_golden_cases']}` "
        f"(`{pct(metadata_coverage_rate)}`, Wilson-95%-BI `{ci_pct(wilson_ci(can['metadata_coverage'], can['total_golden_cases']))}`); "
        f"raw dekking `{raw['metadata_coverage']}/{raw['total_golden_cases']}`.\n"
        "- **Plafond/uitleg:** metadata wordt alleen verwacht voor geschikte documenttypes; raw en canonical worden daarom apart getoond."
    ))
else:
    display(Markdown(f"### Samenvatting\n\n**{golden_classmeta_info.get('status', 'NIET_BEREKEND')}:** {golden_classmeta_info.get('detail', 'Geen detail beschikbaar.')}"))

In [ ]:
consistency_dir = CONSISTENCY_CLASSMETA_DIR
consistency_summary_path = consistency_dir / "consistency_summary.json"

if not consistency_summary_path.exists():
    consistency_classmeta_info = {
        "status": "NIET_GEVONDEN",
        "detail": f"Consistentie-analyse niet gevonden op {consistency_dir}.",
    }
    display(Markdown(f"**NIET_GEVONDEN:** {consistency_classmeta_info['detail']}"))
else:
    consistency_summary = read_json(consistency_summary_path)
    n_docs = consistency_summary.get("document_count_union", 0)
    n_runs = consistency_summary.get("run_count", 0)
    n_unstable = consistency_summary.get("unstable_document_count", 0)
    n_stable = n_docs - n_unstable
    crit_unstable = consistency_summary.get("critical_classification_unstable_count", 0)
    info_unstable = consistency_summary.get("informational_classification_unstable_count", 0)
    meta_unstable = consistency_summary.get("metadata_unstable_count", 0)
    meta_no_evidence = consistency_summary.get("metadata_values_without_evidence_count", 0)
    verifier_corr = consistency_summary.get("verifier_cross_main_correction_count", 0)
    status_unstable = consistency_summary.get("status_unstable_count", 0)
    pct_stable = round(n_stable / n_docs * 100, 1) if n_docs else 0.0

    display(Markdown(f"### Kerncijfers reproduceerbaarheid ({consistency_dir.name})"))
    kern_rows = [
        {"maat": "documenten in test", "waarde": n_docs},
        {"maat": "runs per document", "waarde": n_runs},
        {"maat": "volledig stabiele documenten", "waarde": f"{n_stable}/{n_docs} ({pct_stable}%)"},
        {"maat": "onstabiele documenten", "waarde": n_unstable},
        {"maat": "kritieke classificatievelden onstabiel", "waarde": crit_unstable},
        {"maat": "informatieve classificatievelden onstabiel", "waarde": info_unstable},
        {"maat": "metadatavelden onstabiel", "waarde": meta_unstable},
        {"maat": "metadatawaarden zonder bewijs (audit)", "waarde": meta_no_evidence},
        {"maat": "verifier cross-main correcties", "waarde": verifier_corr},
        {"maat": "documenten met statuswissel", "waarde": status_unstable},
    ]
    display(pd.DataFrame(kern_rows))

    severity_path = consistency_dir / "field_severity_summary.csv"
    if severity_path.exists():
        severity_df = pd.read_csv(severity_path)
        severity_df = severity_df.sort_values(
            ["field_severity", "unstable_count"], ascending=[True, False]
        ).reset_index(drop=True)
        display(Markdown("### Stabiliteit per veld-severity en verschiltype"))
        display(severity_df)
        onstabiel_chart = severity_df[severity_df["unstable_count"] > 0].copy()
        if not onstabiel_chart.empty:
            onstabiel_chart["groep"] = (
                onstabiel_chart["field_severity"] + " / " + onstabiel_chart["difference_type"]
            )
            plot_bar(onstabiel_chart, "groep", "unstable_count",
                     "Onstabiele veld-instanties per severity/verschiltype", color="#E45756")

    def veld_instabiliteit(csv_path: Path, soort: str) -> pd.DataFrame:
        if not csv_path.exists():
            return pd.DataFrame()
        df = pd.read_csv(csv_path)
        df["stable"] = df["stable"].astype(str).str.lower().isin(["true", "1"])
        grp = df.groupby(["field", "field_severity"]).agg(
            documenten=("document_id", "nunique"),
            onstabiel=("stable", lambda s: int((~s).sum())),
        ).reset_index()
        grp.insert(0, "soort", soort)
        return grp

    class_grp = veld_instabiliteit(consistency_dir / "classification_consistency.csv", "classificatie")
    meta_grp = veld_instabiliteit(consistency_dir / "metadata_field_consistency.csv", "metadata")
    veld_instab = pd.concat([class_grp, meta_grp], ignore_index=True)
    if not veld_instab.empty:
        veld_instab = veld_instab[veld_instab["onstabiel"] > 0].sort_values(
            "onstabiel", ascending=False).reset_index(drop=True)
        display(Markdown("### Top instabiele velden (meeste documenten met wisselende waarde)"))
        display(veld_instab.head(25))

    conf_path = consistency_dir / "confidence_consistency.csv"
    if conf_path.exists():
        conf_df = pd.read_csv(conf_path)
        conf_per_veld = conf_df.groupby("field")["stddev"].agg(
            gemiddelde_stddev="mean", hoogste_stddev="max").reset_index()
        conf_per_veld[["gemiddelde_stddev", "hoogste_stddev"]] = (
            conf_per_veld[["gemiddelde_stddev", "hoogste_stddev"]].round(3))
        display(Markdown("### Confidence-spreiding per veld (stddev over 5 runs)"))
        display(conf_per_veld)
        top_conf = conf_df.sort_values("stddev", ascending=False).head(15)[
            ["document_id", "field", "min", "max", "stddev"]].round(3).reset_index(drop=True)
        display(Markdown("### Documenten met grootste confidence-spreiding"))
        display(top_conf)

    unstable_path = consistency_dir / "unstable_documents.json"
    if unstable_path.exists():
        unstable_docs = read_json(unstable_path)
        per_doc_rows = []
        for doc_id, cats in unstable_docs.items():
            cats = cats if isinstance(cats, dict) else {}
            class_items = cats.get("classification", [])
            meta_items = cats.get("metadata", [])
            status_items = cats.get("status", [])
            krit = sum(1 for it in class_items if it.get("field_severity") == "kritiek")
            info = sum(1 for it in class_items if it.get("field_severity") != "kritiek")
            per_doc_rows.append({
                "document_id": doc_id,
                "kritieke_class_velden": krit,
                "informatieve_class_velden": info,
                "metadata_velden": len(meta_items),
                "statuswissel": len(status_items),
                "totaal_onstabiel": krit + info + len(meta_items) + len(status_items),
            })
        per_doc_df = pd.DataFrame(per_doc_rows).sort_values(
            ["kritieke_class_velden", "totaal_onstabiel"], ascending=False).reset_index(drop=True)
        display(Markdown(f"### Per-document instabiliteit ({len(per_doc_df)} onstabiele documenten)"))
        display(per_doc_df)

    consistency_classmeta_info = {
        "status": "WAARSCHUWING" if crit_unstable > 0 else "OK",
        "detail": (
            f"{n_stable}/{n_docs} documenten volledig stabiel ({pct_stable}%); "
            f"{crit_unstable} kritieke classificatievelden en {meta_unstable} metadatavelden "
            f"wisselen tussen {n_runs} identieke runs (bron {consistency_dir.name})."
        ),
    }
    display(Markdown(f"**{consistency_classmeta_info['status']}:** {consistency_classmeta_info['detail']}"))

In [ ]:
import importlib.util as _ilu
import math as _math

_rel_path = PROJECT_ROOT / "pg_database" / "checks" / "kabinetsreactie_validatie" / "reliability.py"
consistency_alpha_overzicht = pd.DataFrame()
if not (CONSISTENCY_CLASSMETA_DIR / "consistency_summary.json").exists() or not _rel_path.exists():
    display(Markdown("**NIET_BEREKEND:** consistentie-data of reliability-module niet gevonden; alpha overgeslagen."))
else:
    _spec = _ilu.spec_from_file_location("kr_reliability", _rel_path)
    _kr = _ilu.module_from_spec(_spec)
    sys.modules["kr_reliability"] = _kr
    _spec.loader.exec_module(_kr)
    pa_fn = _kr.percentage_agreement
    alpha_fn = _kr.krippendorff_alpha_nominal
    RUN_VALUE_COLS = ["run_01_value", "run_02_value", "run_03_value", "run_04_value", "run_05_value"]

    def _norm_val(value):
        if value is None or (isinstance(value, float) and _math.isnan(value)):
            return "<leeg>"
        return str(value)

    def _alpha_per_document(df):
        pas, alphas = [], []
        for _, sub in df.groupby("document_id"):
            runs = [{row["field"]: _norm_val(row[col]) for _, row in sub.iterrows()} for col in RUN_VALUE_COLS]
            pas.append(pa_fn(runs))
            alphas.append(alpha_fn(runs))
        if not alphas:
            return float("nan"), float("nan"), 0
        return float(np.mean(pas)), float(np.mean(alphas)), len(alphas)

    def _alpha_per_field(df):
        rows = []
        for veld, sub in df.groupby("field"):
            runs = [{row["document_id"]: _norm_val(row[col]) for _, row in sub.iterrows()} for col in RUN_VALUE_COLS]
            rows.append({
                "veld": veld,
                "field_severity": sub["field_severity"].iloc[0],
                "documenten": sub["document_id"].nunique(),
                "percentage_overeenstemming": round(pa_fn(runs), 4),
                "krippendorff_alpha": round(alpha_fn(runs), 4),
            })
        return pd.DataFrame(rows)

    class_df_a = pd.read_csv(CONSISTENCY_CLASSMETA_DIR / "classification_consistency.csv")
    meta_df_a = pd.read_csv(CONSISTENCY_CLASSMETA_DIR / "metadata_field_consistency.csv")
    class_krit_df_a = class_df_a[class_df_a["field_severity"] == "kritiek"]

    alpha_rows = []
    for groep, df_a in [
        ("classificatie (kritieke velden)", class_krit_df_a),
        ("classificatie (alle velden)", class_df_a),
        ("metadata", meta_df_a),
    ]:
        pa, alpha, n = _alpha_per_document(df_a)
        alpha_rows.append({
            "groep": groep,
            "eenheid": "per document",
            "n": n,
            "mean_percentage_overeenstemming": round(pa, 4),
            "mean_krippendorff_alpha": round(alpha, 4),
        })
    _ref_row = pd.DataFrame()
    if "reliability_table" in globals() and not reliability_table.empty:
        _ref_row = reliability_table[reliability_table["experiment"] == "parallel_10x3"]
    if not _ref_row.empty:
        alpha_rows.append({
            "groep": "kabinetsreactie-agent (parallel test-hertest)",
            "eenheid": "per document",
            "n": int(_ref_row.iloc[0].get("n_units", 0)) if "n_units" in _ref_row.columns else None,
            "mean_percentage_overeenstemming": round(float(_ref_row.iloc[0]["mean_percentage_agreement"]), 4),
            "mean_krippendorff_alpha": round(float(_ref_row.iloc[0]["mean_krippendorff_alpha"]), 4),
        })
    consistency_alpha_overzicht = pd.DataFrame(alpha_rows)
    display(Markdown("### Overeenstemming en Krippendorff alpha per groep (norm 'substantieel' = 0,667)"))
    display(consistency_alpha_overzicht)

    class_field_alpha = _alpha_per_field(class_df_a).sort_values("krippendorff_alpha").reset_index(drop=True)
    meta_field_alpha = _alpha_per_field(meta_df_a).sort_values("krippendorff_alpha").reset_index(drop=True)
    display(Markdown("### Per-veld alpha - classificatie (laagste alpha eerst)"))
    display(class_field_alpha)
    display(Markdown("### Per-veld alpha - metadata (laagste alpha eerst)"))
    display(meta_field_alpha)

    _krit_alpha = next((r["mean_krippendorff_alpha"] for r in alpha_rows if r["groep"] == "classificatie (kritieke velden)"), None)
    _meta_alpha = next((r["mean_krippendorff_alpha"] for r in alpha_rows if r["groep"] == "metadata"), None)
    if "consistency_classmeta_info" in globals():
        consistency_classmeta_info["alpha_detail"] = (
            f"Krippendorff alpha per document: kritieke classificatie={_krit_alpha}, metadata={_meta_alpha}; "
            f"vergeleken met de kabinetsreactie-referentie uit de reliability-tabel."
        )
        consistency_classmeta_info["detail"] += (
            f" Krippendorff alpha (per doc): kritieke classificatie={_krit_alpha}, metadata={_meta_alpha}."
        )
    _ref_alpha = None if _ref_row.empty else round(float(_ref_row.iloc[0]["mean_krippendorff_alpha"]), 4)
    _ref_text = "niet beschikbaar" if _ref_alpha is None else str(_ref_alpha)
    display(Markdown(
        f"**Vergelijking:** kritieke classificatie alpha = {_krit_alpha}, metadata alpha = {_meta_alpha}. "
        f"De kabinetsreactie-referentie uit de reliability-tabel is {_ref_text}. "
        f"Classificatie en metadata zijn dus duidelijk beter reproduceerbaar dan de verwerkingslabels. "
        f"De informatieve vrije-tekstvelden trekken het gemiddelde van 'alle classificatievelden' omlaag."
    ))

In [ ]:
import importlib.util as _ilu_m
from collections import Counter as _Counter

consistency_majority_overzicht = pd.DataFrame()
consistency_majority_per_veld = pd.DataFrame()
consistency_majority_info = {"status": "NIET_BEREKEND", "detail": "Niet uitgevoerd."}

_cgm_path = (PROJECT_ROOT / "AI agents" / "AI adviescollege documenten - classificatie and metadata"
             / "scripts" / "golden_class_meta" / "compare_golden_db.py")
_cls_csv = CONSISTENCY_CLASSMETA_DIR / "classification_consistency.csv"
_met_csv = CONSISTENCY_CLASSMETA_DIR / "metadata_field_consistency.csv"

if not _cgm_path.exists() or not _cls_csv.exists() or not _met_csv.exists():
    consistency_majority_info = {"status": "NIET_GEVONDEN",
                                 "detail": "Golden-comparator of consistentie-CSV's niet gevonden; majority-test overgeslagen."}
    display(Markdown(f"**NIET_GEVONDEN:** {consistency_majority_info['detail']}"))
else:
    _spec_m = _ilu_m.spec_from_file_location("cgm_majority", _cgm_path)
    _cgm = _ilu_m.module_from_spec(_spec_m)
    sys.modules["cgm_majority"] = _cgm
    _spec_m.loader.exec_module(_cgm)
    _normalize_value = _cgm._normalize_value
    _golden_cases, _ = _cgm._load_golden_cases(_cgm.DEFAULT_GOLDEN_DIR)

    _RUN_COLS = ["run_01_value", "run_02_value", "run_03_value", "run_04_value", "run_05_value"]

    def _parse_run(raw):
        if isinstance(raw, float) and np.isnan(raw):
            return None
        if not isinstance(raw, str):
            return raw
        try:
            return json.loads(raw)
        except Exception:
            return raw

    def _norm_key(value, already_native=False):
        native = value if already_native else _parse_run(value)
        normalized = _normalize_value(native)
        if normalized is None:
            return None
        return json.dumps(normalized, sort_keys=True, ensure_ascii=False)

    _cls_df = pd.read_csv(_cls_csv)
    _met_df = pd.read_csv(_met_csv)
    _cls_fields = set(_cls_df["field"])
    _met_fields = set(_met_df["field"])
    _runmap = {}
    for _df in (_cls_df, _met_df):
        for _, _row in _df.iterrows():
            _runmap[(str(_row["document_id"]), _row["field"])] = [_row[c] for c in _RUN_COLS]
    _consist_docs = {doc for (doc, _) in _runmap}
    _overlap = [doc for doc in _golden_cases if str(doc) in _consist_docs]

    _rows = []
    for _doc in _overlap:
        _gc = _golden_cases[_doc].get("golden_classification") or {}
        _gm = _golden_cases[_doc].get("golden_metadata") or {}
        _plan = [("classificatie", "classification." + f, _gc.get(f))
                 for f in _cgm.CLASSIFICATION_FIELDS if "classification." + f in _cls_fields]
        _plan += [("metadata", "metadata." + f, _gm.get(f))
                  for f in _cgm._metadata_fields_to_compare(_gm) if "metadata." + f in _met_fields]
        for _soort, _field, _gval in _plan:
            _golden_key = _norm_key(_gval, already_native=True)
            if _golden_key is None or (str(_doc), _field) not in _runmap:
                continue
            _runs = [_norm_key(x) for x in _runmap[(str(_doc), _field)]]
            if all(x is None for x in _runs):
                continue
            _single = float(np.mean([1 if x == _golden_key else 0 for x in _runs]))
            _votes = _Counter([x for x in _runs if x is not None])
            _majority = _votes.most_common(1)[0][0] if _votes else None
            _rows.append({"soort": _soort, "veld": _field, "document_id": _doc,
                          "single_run": _single, "majority": 1 if _majority == _golden_key else 0})

    _R = pd.DataFrame(_rows)
    if _R.empty:
        consistency_majority_info = {"status": "NIET_BEREKEND",
                                     "detail": "Geen vergelijkbare velden gevonden tussen golden set en consistentietest."}
        display(Markdown(f"**NIET_BEREKEND:** {consistency_majority_info['detail']}"))
    else:
        def _headline(df_in, groep):
            return {"groep": groep, "veld_instanties": len(df_in),
                    "single_run_accuratesse": round(df_in["single_run"].mean(), 4),
                    "majority_accuratesse": round(df_in["majority"].mean(), 4),
                    "winst_majority": round(df_in["majority"].mean() - df_in["single_run"].mean(), 4)}

        _overzicht_rows = [_headline(_R, "alle velden")]
        for _soort in ("classificatie", "metadata"):
            _sub = _R[_R["soort"] == _soort]
            if not _sub.empty:
                _overzicht_rows.append(_headline(_sub, _soort))
        consistency_majority_overzicht = pd.DataFrame(_overzicht_rows)

        display(Markdown(f"### Majority-vote vs losse run op de golden-overlap (n={len(_overlap)} documenten)"))
        display(consistency_majority_overzicht)

        consistency_majority_per_veld = (
            _R.groupby(["soort", "veld"]).agg(
                documenten=("document_id", "nunique"),
                single_run_accuratesse=("single_run", "mean"),
                majority_accuratesse=("majority", "mean"),
            ).reset_index()
        )
        consistency_majority_per_veld["winst_majority"] = (
            consistency_majority_per_veld["majority_accuratesse"] - consistency_majority_per_veld["single_run_accuratesse"])
        for _c in ["single_run_accuratesse", "majority_accuratesse", "winst_majority"]:
            consistency_majority_per_veld[_c] = consistency_majority_per_veld[_c].round(3)
        consistency_majority_per_veld = consistency_majority_per_veld.sort_values("winst_majority", ascending=False).reset_index(drop=True)
        display(Markdown("### Per veld: winst van majority-vote (grootste winst eerst)"))
        display(consistency_majority_per_veld)

        _single_all = consistency_majority_overzicht.loc[0, "single_run_accuratesse"]
        _maj_all = consistency_majority_overzicht.loc[0, "majority_accuratesse"]
        _winst = round(_maj_all - _single_all, 4)
        consistency_majority_info = {
            "status": "OK",
            "detail": (f"Op {len(_overlap)} golden-overlap documenten: losse run {_single_all} -> majority-vote {_maj_all} "
                       f"(winst {_winst}). Majority helpt vooral bij vrije-tekst-metadata (titel, advies_aanvrager); "
                       f"systematische classificatiefouten corrigeert het niet."),
        }
        display(Markdown(
            f"**{consistency_majority_info['status']}:** majority-vote tilt de accuratesse van {_single_all} naar {_maj_all} "
            f"(+{_winst}). Een bescheiden maar consistente winst: majority-vote middelt willekeurige ruis weg, maar kan "
            f"systematische fouten (waar alle runs dezelfde kant op leunen) niet repareren. Het sterkste argument blijft "
            f"betrouwbaarheid: majority-of-5 geeft een vaste, reproduceerbare keuze in plaats van een loterij per run."
        ))

In [ ]:
old_advies_summary = read_json(OLD_ADVIES_SUMMARY_JSON)
old_kabinet_summary = read_json(OLD_KABINET_SUMMARY_JSON)

KABINET_OLD_SOURCE_LABEL = "oude 39-doc validatiebatch / thesis-index"
KABINET_CURRENT_SOURCE_LABEL = f"actuele volledige batch {KABINET_BATCH_DIR.name}"


def claim_row(claim: str, old_value: Any, current_value: Any, source_old: str, source_current: str, action: str, severity: str = "midden") -> dict[str, Any]:
    status = status_exact(old_value, current_value)
    return {
        "claim": claim,
        "oude waarde": old_value,
        "actuele waarde": current_value,
        "status": status,
        "actie voor tekst": action if status != "OK" else "Laat staan.",
        "bron oude waarde": source_old,
        "bron actuele waarde": source_current,
        "ernst": severity,
    }


claim_rows = [
    claim_row(
        "Adviesvalidatie aantal documenten",
        1264,
        advies_summary.get("n_documenten"),
        "geplakte/oudere thesis-tekst",
        "live adviesvalidatie include_english=False",
        "Vervang documentaantal door live waarde of markeer peildatum expliciet.",
        "hoog",
    ),
    claim_row(
        "Adviesvalidatie groen",
        779,
        advies_summary.get("n_groen"),
        "geplakte/oudere thesis-tekst",
        "live adviesvalidatie include_english=False",
        "Vervang groen-aantal door live waarde.",
        "hoog",
    ),
    claim_row(
        "Adviesvalidatie oranje",
        459,
        advies_summary.get("n_oranje"),
        "geplakte/oudere thesis-tekst",
        "live adviesvalidatie include_english=False",
        "Vervang oranje-aantal door live waarde.",
        "hoog",
    ),
    claim_row(
        "Adviesvalidatie rood",
        26,
        advies_summary.get("n_rood"),
        "geplakte/oudere thesis-tekst",
        "live adviesvalidatie include_english=False",
        "Vervang rood-aantal door live waarde als dit wijzigt.",
    ),
    claim_row(
        "Robuustheidsset groen+oranje",
        1238,
        None if advies_summary.get("n_groen") is None else advies_summary.get("n_groen") + advies_summary.get("n_oranje"),
        "geplakte/oudere thesis-tekst",
        "live adviesvalidatie include_english=False",
        "Vervang groen+oranje door live som.",
        "hoog",
    ),
    claim_row(
        "Kabinetsreactie documenten",
        39,
        kabinet_summary.get("n_documenten"),
        KABINET_OLD_SOURCE_LABEL,
        KABINET_CURRENT_SOURCE_LABEL,
        "Formuleer 39 alleen als historische validatiesample; gebruik voor de actuele analysebatch de live waarde.",
        "hoog",
    ),
    claim_row(
        "Kabinetsreactie advieselementen",
        1218,
        kabinet_summary.get("n_items"),
        KABINET_OLD_SOURCE_LABEL,
        KABINET_CURRENT_SOURCE_LABEL,
        "Vervang itemaantal door live waarde of benoem expliciet dat 1218 bij de oude 39-doc sample hoort.",
        "hoog",
    ),
    claim_row(
        "Kabinetsreactie groen",
        20,
        kabinet_summary.get("n_groen"),
        KABINET_OLD_SOURCE_LABEL,
        KABINET_CURRENT_SOURCE_LABEL,
        "Vervang groen-aantal door live waarde of splits oude sample en actuele batch.",
        "hoog",
    ),
    claim_row(
        "Kabinetsreactie oranje",
        19,
        kabinet_summary.get("n_oranje"),
        KABINET_OLD_SOURCE_LABEL,
        KABINET_CURRENT_SOURCE_LABEL,
        "Vervang oranje-aantal door live waarde of splits oude sample en actuele batch.",
        "hoog",
    ),
    claim_row(
        "Kabinetsreactie rood",
        0,
        kabinet_summary.get("n_rood"),
        KABINET_OLD_SOURCE_LABEL,
        KABINET_CURRENT_SOURCE_LABEL,
        "Vervang rood-aantal door live waarde of splits oude sample en actuele batch.",
    ),
    claim_row(
        "Items met >=3 trace warnings",
        1003,
        kabinet_summary.get("items_met_3plus_trace_warnings"),
        KABINET_OLD_SOURCE_LABEL,
        KABINET_CURRENT_SOURCE_LABEL,
        "Vervang trace-warning aantal door live waarde en behoud caveat.",
        "hoog",
    ),
]

for experiment, old_pa, old_alpha in [
    ("parallel_10x3", 0.2537, 0.1847),
    ("sequentieel_4907", 0.1875, 0.2027),
    ("seed_4907", 0.125, 0.027),
]:
    rel_row = reliability_table[reliability_table["experiment"] == experiment]
    current_pa = None if rel_row.empty else rel_row.iloc[0]["mean_percentage_agreement"]
    current_alpha = None if rel_row.empty else rel_row.iloc[0]["mean_krippendorff_alpha"]
    claim_rows.append({
        "claim": f"Reliability {experiment} PA",
        "oude waarde": old_pa,
        "actuele waarde": current_pa,
        "status": status_float(old_pa, current_pa, tolerance=0.0001),
        "actie voor tekst": "Laat staan als oude JSON-bron bewust wordt gebruikt; dit is geen nieuwe run.",
        "bron oude waarde": "oude reliability samenvatting",
        "bron actuele waarde": "ingelezen reliability JSON",
        "ernst": "laag",
    })
    claim_rows.append({
        "claim": f"Reliability {experiment} alpha",
        "oude waarde": old_alpha,
        "actuele waarde": current_alpha,
        "status": status_float(old_alpha, current_alpha, tolerance=0.0001),
        "actie voor tekst": "Laat staan als oude JSON-bron bewust wordt gebruikt; dit is geen nieuwe run.",
        "bron oude waarde": "oude reliability samenvatting",
        "bron actuele waarde": "ingelezen reliability JSON",
        "ernst": "laag",
    })

claim_table = pd.DataFrame(claim_rows)
claim_table

In [ ]:
manual_claims = pd.DataFrame([
    {"claim": "Volledigheid van de volledige Kaderwet-populatie", "status": "HANDMATIG", "actie": "Alleen claimen met aparte corpusverantwoording."},
    {"claim": "Recall over alle adviezen, reacties en koppelingen", "status": "HANDMATIG", "actie": "Niet als bewezen formuleren; geen onafhankelijke volledige gouden standaard."},
    {"claim": "Inhoudelijke juistheid van alle AI-classificaties", "status": "HANDMATIG", "actie": "Validatie toetst intern; inhoudelijke juistheid vraagt handmatige steekproef."},
    {"claim": "Causale invloed van externe inbreng op verwerking", "status": "NIET_TOETSBAAR", "actie": "Als associatief formuleren, niet causaal."},
    {"claim": "Informele beleidsimpact buiten formele documenten", "status": "NIET_TOETSBAAR", "actie": "Buiten bereik van meetinstrument houden."},
])
manual_claims

In [ ]:
summary_rows = [
    {"onderdeel": "databronnen", "status": "OK" if inputcheck["bestaat"].all() else "AANPASSEN", "detail": f"Peildatum {DATA_PEILDATUM}; {int(inputcheck['bestaat'].sum())}/{len(inputcheck)} bronnen gevonden."},
    {"onderdeel": "bronbundel thesis", "status": bronmatrix_info.get("status", "NIET_BEREKEND"), "detail": bronmatrix_info.get("detail", "Geen detail beschikbaar.")},
    {"onderdeel": "DV1 corpuscontext", "status": "OK" if DV1_COLLEGE_DEFINITIES_CSV.exists() and DV1_ANALYSEPOPULATIES_CSV.exists() else "NIET_BEREKEND", "detail": f"Canonieke DV1-scope: {DV1_CORPUS_COLLEGES_EXPECTED} collegefasen; oude contextbundel: {DV1_OUTPUT_DIR.name}."},
    {"onderdeel": "adviesvalidatie", "status": "OK" if advies_error is None else "NIET_BEREKEND", "detail": advies_error or f"{advies_summary.get('n_documenten')} documenten, NL-only"},
    {"onderdeel": "kabinetsreactievalidatie", "status": "OK" if kabinet_error is None else "NIET_BEREKEND", "detail": kabinet_error or f"{kabinet_summary.get('n_documenten')} documenten, {kabinet_summary.get('n_items')} items; 171-filter via {KABINET_171_FROZEN_DIR.name}"},
    {"onderdeel": "historische 39-batch", "status": "OK" if historical_result_json_count == EXPECTED_HISTORISCHE_KABINET_RESULT_JSONS else "AANPASSEN", "detail": f"Alleen referentie: {historical_result_json_count}/{EXPECTED_HISTORISCHE_KABINET_RESULT_JSONS} result-jsons."},
    {"onderdeel": "bronconflicten kabinetrapport", "status": "AANPASSEN" if (not bronconflict.empty and (bronconflict["status"] == "BRONCONFLICT").any()) else "OK", "detail": "Gebruik live cijfers/machine-JSON bij conflict."},
    {"onderdeel": "reliability", "status": "OK", "detail": "Alleen bestaande JSONs ingelezen; geen nieuwe reliability-run."},
    {"onderdeel": "golden-set kabinetsreactie", "status": kabinetsreactie_golden_set_info.get("status", "NIET_BEREKEND"), "detail": kabinetsreactie_golden_set_info.get("detail", "Geen detail beschikbaar.")},
    {"onderdeel": "confusion matrix kabinetsreactie", "status": kabinetsreactie_confusion_info.get("status", "NIET_BEREKEND"), "detail": kabinetsreactie_confusion_info.get("detail", "Geen detail beschikbaar.")},
    {"onderdeel": "non-random dekking/missingness", "status": kabinetsreactie_nonrandom_info.get("status", "NIET_BEREKEND"), "detail": kabinetsreactie_nonrandom_info.get("detail", "Geen detail beschikbaar.")},
    {"onderdeel": "kabinetsreactie retry-wisselaars", "status": kabinetsreactie_retry_wisselaars_info.get("status", "NIET_BEREKEND"), "detail": kabinetsreactie_retry_wisselaars_info.get("detail", "Geen detail beschikbaar.")},
    {"onderdeel": "adviesextractie retry-info", "status": adviesextractie_retry_info.get("status", "NIET_BEREKEND"), "detail": adviesextractie_retry_info.get("detail", "Geen detail beschikbaar.")},
    {"onderdeel": "extra robuustheidstests", "status": "OK" if ('robustness_summary' in globals() and not robustness_summary.empty and not (robustness_summary["status"] == "WAARSCHUWING").any()) else "WAARSCHUWING", "detail": "Zie robuustheidssamenvatting; niet-uitvoerbare tests betekenen ontbrekende kolommen."},
    {"onderdeel": "claim-checks", "status": "AANPASSEN" if (claim_table["status"] == "AANPASSEN").any() else "OK", "detail": f"{int((claim_table['status'] == 'AANPASSEN').sum())} claims wijken af."},
]

if "golden_dekking_info" in globals():
    summary_rows.append({
        "onderdeel": "golden-set adviesextractie",
        "status": golden_dekking_info.get("status", "NIET_BEREKEND"),
        "detail": golden_dekking_info.get("detail", "Geen detail beschikbaar."),
    })

if "golden_classmeta_info" in globals():
    summary_rows.append({
        "onderdeel": "golden-set classificatie & metadata",
        "status": golden_classmeta_info.get("status", "NIET_BEREKEND"),
        "detail": golden_classmeta_info.get("detail", "Geen detail beschikbaar."),
    })

if "consistency_classmeta_info" in globals():
    summary_rows.append({
        "onderdeel": "reproduceerbaarheid classificatie & metadata (100x5)",
        "status": consistency_classmeta_info.get("status", "NIET_BEREKEND"),
        "detail": consistency_classmeta_info.get("detail", "Geen detail beschikbaar."),
    })

if "consistency_majority_info" in globals():
    summary_rows.append({
        "onderdeel": "majority-vote vs losse run (golden-overlap)",
        "status": consistency_majority_info.get("status", "NIET_BEREKEND"),
        "detail": consistency_majority_info.get("detail", "Geen detail beschikbaar."),
    })

eindoverzicht = pd.DataFrame(summary_rows)
display(eindoverzicht)

display(Markdown("### Claims die tekstactie vragen"))
claim_filter = claim_table["status"].isin(["AANPASSEN", "NIET_BEREKEND"])
display(claim_table.loc[claim_filter])

In [ ]:
validatieopzet_rows = [
    proportion_record("gevonden inputbronnen", int(inputcheck["bestaat"].sum()), len(inputcheck)),
    proportion_record("gevonden bronmatrix-items", int(bronmatrix["bestaat"].sum()), len(bronmatrix)),
]
validatieopzet_table = pd.DataFrame(validatieopzet_rows)
display(Markdown(
    f"Peildatum `{DATA_PEILDATUM}`. De notebook gebruikt vaste bronpaden en schrijft zelf geen bestanden. "
    f"Adviesdata wordt geladen met `include_english={INCLUDE_ENGLISH}`."
))
display(validatieopzet_table)

In [ ]:
golden_accuracy_rows = []
if "kabinetsreactie_accuracy_ci" in globals() and not kabinetsreactie_accuracy_ci.empty:
    golden_accuracy_rows.extend(kabinetsreactie_accuracy_ci.assign(pijplijn="kabinetsreactie-agent").to_dict("records"))
if "classmeta_reports" in globals() and classmeta_reports:
    can = classmeta_reports["canonical"]["summary"]
    golden_accuracy_rows.append({"pijplijn": "classificatie & metadata", **proportion_record("classificatie exact-match", can["classification_matches"], can["classification_coverage"])})
    golden_accuracy_rows.append({"pijplijn": "classificatie & metadata", **proportion_record("metadata-dekking canonical", can["metadata_coverage"], can["total_golden_cases"])})
if "golden_scores_data" in globals() and golden_scores_data:
    for metric, stats in golden_scores_data.items():
        if metric.endswith("_f1"):
            golden_accuracy_rows.append({
                "pijplijn": "adviesextractie",
                "maat": metric.replace("_", "-"),
                "teller": None,
                "noemer": stats.get("n"),
                "waarde": f"{stats.get('mean'):.3f}",
                "wilson_95bi": "n.v.t.; zie bootstrap in scoretabel",
            })
golden_accuracy_table = pd.DataFrame(golden_accuracy_rows)
display(Markdown("Golden-setmaten worden met Wilson-95%-BI getoond waar teller/noemer bekend zijn. F1-maten gebruiken lichte bootstrap in de scoretabellen."))
display(golden_accuracy_table)

In [ ]:
display(Markdown(
    f"Advieslabels: `{advies_summary.get('n_groen')}` groen, `{advies_summary.get('n_oranje')}` oranje, `{advies_summary.get('n_rood')}` rood. "
    f"Kabinetsreactielabels: `{kabinet_summary.get('n_groen')}` groen, `{kabinet_summary.get('n_oranje')}` oranje, `{kabinet_summary.get('n_rood')}` rood."
))
if "kabinetsreactie_label_metrics" in globals() and not kabinetsreactie_label_metrics.empty:
    display(Markdown("### Kabinetsreactie: per-label fouten en onzekerheid"))
    display(kabinetsreactie_label_metrics)
if advies_flags is not None and not advies_flags.empty:
    display(Markdown("### Adviesvalidatie: meest voorkomende checks"))
    display(value_counts_table(advies_flags, "check_naam", "check_naam").head(10))

In [ ]:
entropy_rows = []

if "advies_labels" in globals() and advies_labels is not None and not advies_labels.empty and "label" in advies_labels.columns:
    entropy_rows.append(label_entropy_record("adviesvalidatie groen/oranje/rood", advies_labels["label"], ["groen", "oranje", "rood"]))

if "kabinet_labels" in globals() and kabinet_labels is not None and not kabinet_labels.empty and "label" in kabinet_labels.columns:
    entropy_rows.append(label_entropy_record("kabinetsreactievalidatie groen/oranje/rood", kabinet_labels["label"], ["groen", "oranje", "rood"]))

if "kabinet_data" in globals() and kabinet_data is not None and hasattr(kabinet_data, "df_items") and "finale_verwerkingslabel" in kabinet_data.df_items.columns:
    expected_processing_labels = [
        "overgenomen",
        "deels_overgenomen",
        "herformuleerd",
        "gerelativeerd",
        "afgewezen",
        "onduidelijk",
        "niet_herkenbaar_verwerkt",
        "niet_behandeld",
    ]
    entropy_rows.append(label_entropy_record("kabinetsreactie verwerkingslabels", kabinet_data.df_items["finale_verwerkingslabel"], expected_processing_labels))

if "kabinetsreactie_golden_label_eval" in globals() and not kabinetsreactie_golden_label_eval.empty:
    if "coarse_truth" in kabinetsreactie_golden_label_eval.columns:
        entropy_rows.append(label_entropy_record("golden-set waarheid grove labels", kabinetsreactie_golden_label_eval["coarse_truth"]))
    if "ai_coarse" in kabinetsreactie_golden_label_eval.columns:
        entropy_rows.append(label_entropy_record("AI-voorspelling grove labels", kabinetsreactie_golden_label_eval["ai_coarse"]))

label_entropy_table = pd.DataFrame(entropy_rows)
label_entropy_info = {
    "status": "NIET_BEREKEND" if label_entropy_table.empty else ("WAARSCHUWING" if (label_entropy_table["status"] == "WAARSCHUWING").any() else ("LET_OP" if (label_entropy_table["status"] == "LET_OP").any() else "OK")),
    "detail": "Geen label-entropietests berekend." if label_entropy_table.empty else f"{len(label_entropy_table)} labelverdelingen getest; laagste genormaliseerde entropie: {label_entropy_table['genormaliseerde_entropy'].dropna().min():.3f}.",
}

display(Markdown(
    "Deze tabel gebruikt Shannon-entropie. `genormaliseerde_entropy` loopt van 0 (volledige concentratie) tot 1 (maximale spreiding over de opgegeven/gevonden labels)."
))
display(label_entropy_table)
label_entropy_info

In [ ]:
stabiliteit_rows = []
for name, info in [
    ("kabinetsreactie reliability", {"status": "OK" if "reliability_table" in globals() and not reliability_table.empty else "NIET_BEREKEND", "detail": "bestaande reliability-JSONs ingelezen"}),
    ("kabinetsreactie retry-wisselaars", globals().get("kabinetsreactie_retry_wisselaars_info", {})),
    ("adviesextractie retry-info", globals().get("adviesextractie_retry_info", {})),
    ("classificatie & metadata reproduceerbaarheid", globals().get("consistency_classmeta_info", {})),
    ("majority-vote golden-overlap", globals().get("consistency_majority_info", {})),
]:
    stabiliteit_rows.append({"controle": name, "status": info.get("status", "NIET_BEREKEND"), "detail": info.get("detail", "Geen detail beschikbaar.")})
stabiliteit_table = pd.DataFrame(stabiliteit_rows)
display(stabiliteit_table)

In [ ]:
dekking_rows = []
for name, info in [
    ("bronbundel", globals().get("bronmatrix_info", {})),
    ("adviesextractie golden-set dekking", globals().get("golden_dekking_info", {})),
    ("kabinetsreactie non-random missingness", globals().get("kabinetsreactie_nonrandom_info", {})),
    ("classificatie & metadata golden-set", globals().get("golden_classmeta_info", {})),
]:
    dekking_rows.append({"controle": name, "status": info.get("status", "NIET_BEREKEND"), "detail": info.get("detail", "Geen detail beschikbaar.")})
dekking_table = pd.DataFrame(dekking_rows)
display(dekking_table)
if "kabinetsreactie_nonrandom_tables" in globals() and "live_trace_per_label" in kabinetsreactie_nonrandom_tables:
    display(Markdown("### Trace warnings per label"))
    display(kabinetsreactie_nonrandom_tables["live_trace_per_label"])

In [ ]:
if "robustness_summary" in globals() and not robustness_summary.empty:
    display(Markdown("Robuustheidstests tonen of conclusies gevoelig zijn voor selectie, periode, groepsverschillen, proxies of outliers."))
    display(robustness_summary)
else:
    display(Markdown("**NIET_BEREKEND:** geen robuustheidssamenvatting beschikbaar."))

In [ ]:
trace_rows = []
if "s5_boxcontrole" in globals() and not s5_boxcontrole.empty:
    oordeel_col = "handmatige_boxcontrole" if "handmatige_boxcontrole" in s5_boxcontrole.columns else "oordeel" if "oordeel" in s5_boxcontrole.columns else None
    if oordeel_col is not None:
        supports = int((s5_boxcontrole[oordeel_col] == "SUPPORTS").sum())
        trace_rows.append(proportion_record("S5-boxcontrole SUPPORTS", supports, len(s5_boxcontrole)))
if "bronconflict" in globals() and not bronconflict.empty:
    conflicts = int((bronconflict["status"] == "BRONCONFLICT").sum())
    trace_rows.append(proportion_record("bronconflicten oude rapporttekst", conflicts, len(bronconflict)))
trace_table = pd.DataFrame(trace_rows)
display(Markdown("Broncontrole geeft voorrang aan machine-JSON en live berekende tabellen boven oude markdownrapporten."))
display(trace_table)
if "claim_table" in globals() and not claim_table.empty:
    display(Markdown("### Claims die tekstactie vragen"))
    display(claim_table.loc[claim_table["status"].isin(["AANPASSEN", "NIET_BEREKEND"])])

In [ ]:
def _status_value(info: dict[str, Any] | str | None) -> str:
    if isinstance(info, dict):
        return str(info.get("status", "NIET_BEREKEND"))
    if info is None:
        return "NIET_BEREKEND"
    return str(info)

def _pipeline_oordeel(statuses: list[str], voorzichtig_signaal: bool = False, verkennend_signaal: bool = False) -> str:
    blocking = {"FOUT", "AANPASSEN"}
    incomplete = {"NIET_BEREKEND", "NIET_GEVONDEN", "NIET_UITVOERBAAR"}
    if verkennend_signaal or any(status in blocking for status in statuses):
        return "verkennend"
    if voorzichtig_signaal or any(status in incomplete or status == "WAARSCHUWING" for status in statuses):
        return "voorzichtig"
    return "geschikt"

advies_zwakke_subvelden = False
if "golden_scores_data" in globals() and golden_scores_data:
    zwakke_metric_names = ["consultation_actor_f1", "report_label_accuracy", "policy_logic_link_f1"]
    advies_zwakke_subvelden = any(golden_scores_data.get(metric, {}).get("mean", 1.0) < 0.5 for metric in zwakke_metric_names if metric in golden_scores_data)

kabinet_lage_accuracy = False
if "kabinetsreactie_confusion_info" in globals():
    kabinet_lage_accuracy = kabinetsreactie_confusion_info.get("accuracy_pct") is not None and kabinetsreactie_confusion_info.get("accuracy_pct") < 67

classmeta_alpha_warning = False
if "consistency_classmeta_info" in globals() and isinstance(consistency_classmeta_info, dict):
    classmeta_alpha_warning = consistency_classmeta_info.get("status") == "WAARSCHUWING"

entropy_warning = False
if "label_entropy_info" in globals() and isinstance(label_entropy_info, dict):
    entropy_warning = label_entropy_info.get("status") in {"WAARSCHUWING", "LET_OP"}

pipeline_rows = [
    {"pijplijn": "adviesextractie", "oordeel": _pipeline_oordeel([_status_value(globals().get("golden_dekking_info")), _status_value(globals().get("golden_scores_info")), _status_value(globals().get("adviesextractie_retry_info"))], voorzichtig_signaal=advies_zwakke_subvelden), "basis": "golden-set scores, dekking en retry/test-hertest status", "gebruik": "kerninhoud bruikbaar; zwakke subvelden apart nuanceren"},
    {"pijplijn": "kabinetsreactie-agent", "oordeel": _pipeline_oordeel(["OK" if kabinet_error is None else "NIET_BEREKEND", _status_value(globals().get("kabinetsreactie_golden_set_info")), _status_value(globals().get("kabinetsreactie_confusion_info")), _status_value(globals().get("kabinetsreactie_nonrandom_info")), _status_value(globals().get("kabinetsreactie_retry_wisselaars_info"))], voorzichtig_signaal=(kabinet_lage_accuracy or entropy_warning)), "basis": "golden-set labels, confusion matrix, entropy, missingness en retry-wisselaars", "gebruik": "geschikt voor transparante controle; verwerkingslabels voorzichtig interpreteren"},
    {"pijplijn": "classificatie & metadata", "oordeel": _pipeline_oordeel([_status_value(globals().get("golden_classmeta_info")), _status_value(globals().get("consistency_classmeta_info")), _status_value(globals().get("consistency_majority_info"))], voorzichtig_signaal=classmeta_alpha_warning), "basis": "golden-set tegen database en reproduceerbaarheid over herhaalde runs", "gebruik": "bruikbaar voor corpusstructuur; metadata-dekking blijft expliciet begrensd"},
    {"pijplijn": "traceerbaarheid en broncontrole", "oordeel": _pipeline_oordeel([_status_value(globals().get("bronmatrix_info")), "AANPASSEN" if ("bronconflict" in globals() and not bronconflict.empty and (bronconflict["status"] == "BRONCONFLICT").any()) else "OK"], voorzichtig_signaal=("s5_boxcontrole" in globals() and not s5_boxcontrole.empty)), "basis": "bronmatrix, machine-JSON versus rapporttekst en S5-boxcontrole", "gebruik": "altijd bronverwijzing en caveat opnemen bij trace warnings"},
]
pijplijn_eindoordeel = pd.DataFrame(pipeline_rows)
display(pijplijn_eindoordeel)